In [30]:
import os
import json
import pandas as pd
import numpy as np

folder_path = r"D:\real project DA dataset\2027 wc\odis_male_json"

json_files = [
    file for file in os.listdir(folder_path)
    if file.endswith(".json")
]

print("Total JSON files:", len(json_files))

Total JSON files: 2565


In [2]:
records = []

for file in json_files:
    file_path = os.path.join(folder_path, file)

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    info = data["info"]

    records.append({
        "file": file,
        "match_id": info.get("match_type_number"),
        "date": info["dates"][0],
        "gender": info.get("gender"),
        "match_type": info.get("match_type"),
        "venue": info.get("venue"),
        "city": info.get("city"),
        "team1": info["teams"][0],
        "team2": info["teams"][1],
        "winner": info.get("outcome", {}).get("winner")
    })

matches = pd.DataFrame(records)

matches.head()

,file,match_id,date,gender,match_type,venue,city,team1,team2,winner
0,1000887.json,3817,2017-01-13,male,ODI,"Brisbane Cricket Ground, Woolloongabba",Brisbane,Australia,Pakistan,Australia
1,1000889.json,3818,2017-01-15,male,ODI,Melbourne Cricket Ground,None,Australia,Pakistan,Pakistan
2,1000891.json,3820,2017-01-19,male,ODI,Western Australia Cricket Association Ground,Perth,Australia,Pakistan,Australia
3,1000893.json,3822,2017-01-22,male,ODI,Sydney Cricket Ground,None,Australia,Pakistan,Australia
4,1000895.json,3826,2017-01-26,male,ODI,Adelaide Oval,None,Australia,Pakistan,Australia


In [3]:
print("Rows:", len(matches))
print("Columns:", matches.columns.tolist())

Rows: 2565
Columns: ['file', 'match_id', 'date', 'gender', 'match_type', 'venue', 'city', 'team1', 'team2', 'winner']


In [4]:
venue_counts = (
    matches["venue"]
    .value_counts()
    .reset_index()
)

venue_counts.columns = ["venue", "matches"]

venue_counts

,venue,matches
0,Harare Sports Club,113
1,Shere Bangla National Stadium,85
2,Dubai International Cricket Stadium,58
3,Sydney Cricket Ground,48
4,R Premadasa Stadium,47
...,...,...
307,Wanderers Cricket Ground,1
308,Goodyear Park,1
309,Keenan Stadium,1
310,Nahar Singh Stadium,1


In [5]:
# Show venues containing keywords related to South Africa, Zimbabwe and Namibia

keywords = [
    "South Africa",
    "Johannesburg",
    "Wanderers",
    "Centurion",
    "Newlands",
    "Cape Town",
    "Durban",
    "Kingsmead",
    "Gqeberha",
    "Port Elizabeth",
    "Bloemfontein",
    "Mangaung",
    "Paarl",
    "Boland",
    "East London",
    "Buffalo",
    "Zimbabwe",
    "Harare",
    "Bulawayo",
    "Victoria Falls",
    "Namibia",
    "Windhoek"
]

venue_list = matches["venue"].dropna().unique()

relevant_venues = [
    venue for venue in venue_list
    if any(keyword.lower() in venue.lower() for keyword in keywords)
]

for venue in sorted(relevant_venues):
    count = (matches["venue"] == venue).sum()
    print(f"{venue} : {count}")

Boland Bank Park, Paarl : 3
Boland Park : 5
Boland Park, Paarl : 4
Buffalo Park : 5
Buffalo Park, East London : 4
Bulawayo Athletic Club : 6
Goodyear Park, Bloemfontein : 5
Harare Sports Club : 113
Kingsmead : 23
Kingsmead, Durban : 5
Mangaung Oval : 4
Mangaung Oval, Bloemfontein : 4
Namibia Cricket Ground, Windhoek : 5
New Wanderers Stadium : 19
New Wanderers Stadium, Johannesburg : 5
Newlands : 19
Newlands, Cape Town : 7
Queens Sports Club, Bulawayo : 17
St George's Park, Gqeberha : 1
St George's Park, Port Elizabeth : 5
SuperSport Park, Centurion : 11
Takashinga Sports Club, Highfield, Harare : 9
The Wanderers Stadium : 7
The Wanderers Stadium, Johannesburg : 5
United Cricket Club Ground, Windhoek : 16
Wanderers Cricket Ground : 1
Wanderers Cricket Ground, Windhoek : 20


In [6]:
venue_city = (
    matches.groupby(["venue", "city"])
    .size()
    .reset_index(name="matches")
    .sort_values(["city", "venue"])
)

venue_city

,venue,city,matches
150,Mannofield Park,Aberdeen,8
151,"Mannofield Park, Aberdeen",Aberdeen,5
241,Sheikh Zayed Stadium,Abu Dhabi,37
310,"Zayed Cricket Stadium, Abu Dhabi",Abu Dhabi,3
1,Adelaide Oval,Adelaide,6
...,...,...,...
2,Affies Park,Windhoek,1
168,"Namibia Cricket Ground, Windhoek",Windhoek,5
283,"United Cricket Club Ground, Windhoek",Windhoek,16
294,Wanderers Cricket Ground,Windhoek,1


In [7]:
countries_cities = venue_city[
    venue_city["city"].isin([
        "Johannesburg",
        "Centurion",
        "Cape Town",
        "Durban",
        "Gqeberha",
        "Port Elizabeth",
        "Bloemfontein",
        "Paarl",
        "East London",
        "Harare",
        "Bulawayo",
        "Victoria Falls",
        "Windhoek"
    ])
]

countries_cities

,venue,city,matches
53,Chevrolet Park,Bloemfontein,2
86,Goodyear Park,Bloemfontein,1
87,"Goodyear Park, Bloemfontein",Bloemfontein,5
148,Mangaung Oval,Bloemfontein,4
149,"Mangaung Oval, Bloemfontein",Bloemfontein,4
188,OUTsurance Oval,Bloemfontein,2
41,Bulawayo Athletic Club,Bulawayo,5
205,Queens Sports Club,Bulawayo,37
206,"Queens Sports Club, Bulawayo",Bulawayo,17
183,Newlands,Cape Town,19


In [8]:
candidate_venues = [
    "New Wanderers Stadium",
    "New Wanderers Stadium, Johannesburg",
    "The Wanderers Stadium",
    "The Wanderers Stadium, Johannesburg",

    "SuperSport Park",
    "SuperSport Park, Centurion",

    "Newlands",
    "Newlands, Cape Town",

    "Kingsmead",
    "Kingsmead, Durban",

    "St George's Park",
    "St George's Park, Gqeberha",
    "St George's Park, Port Elizabeth",

    "Mangaung Oval",
    "Mangaung Oval, Bloemfontein",
    "Goodyear Park",
    "Goodyear Park, Bloemfontein",
    "Chevrolet Park",
    "OUTsurance Oval",

    "Boland Bank Park, Paarl",
    "Boland Park",
    "Boland Park, Paarl",

    "Buffalo Park",
    "Buffalo Park, East London",

    "Harare Sports Club",
    "Queens Sports Club",
    
    "Namibia Cricket Ground, Windhoek"
]

candidate_counts = (
    matches[matches["venue"].isin(candidate_venues)]
    .groupby(["venue", "city"])
    .size()
    .reset_index(name="matches")
    .sort_values("city")
)

candidate_counts

,venue,city,matches
5,Chevrolet Park,Bloemfontein,2
6,Goodyear Park,Bloemfontein,1
7,"Goodyear Park, Bloemfontein",Bloemfontein,5
18,OUTsurance Oval,Bloemfontein,2
11,Mangaung Oval,Bloemfontein,4
12,"Mangaung Oval, Bloemfontein",Bloemfontein,4
19,Queens Sports Club,Bulawayo,37
17,"Newlands, Cape Town",Cape Town,7
16,Newlands,Cape Town,19
24,"SuperSport Park, Centurion",Centurion,11


In [9]:
matches["date"] = pd.to_datetime(matches["date"])

candidate_2015 = (
    matches[
        (matches["date"] >= "2015-01-01") &
        (matches["venue"].isin(candidate_venues))
    ]
    .groupby(["venue", "city"])
    .size()
    .reset_index(name="matches_since_2015")
    .sort_values("city")
)

candidate_2015

,venue,city,matches_since_2015
6,Mangaung Oval,Bloemfontein,3
7,"Mangaung Oval, Bloemfontein",Bloemfontein,4
12,Queens Sports Club,Bulawayo,13
10,Newlands,Cape Town,7
11,"Newlands, Cape Town",Cape Town,2
16,"SuperSport Park, Centurion",Centurion,6
15,SuperSport Park,Centurion,9
5,Kingsmead,Durban,8
2,Buffalo Park,East London,2
3,"Buffalo Park, East London",East London,1


In [10]:
venue_mapping = {

    "Mangaung Oval": "Mangaung Oval",
    "Mangaung Oval, Bloemfontein": "Mangaung Oval",

    "Queens Sports Club": "Queens Sports Club",

    "Newlands": "Newlands",
    "Newlands, Cape Town": "Newlands",

    "SuperSport Park": "SuperSport Park",
    "SuperSport Park, Centurion": "SuperSport Park",

    "Kingsmead": "Kingsmead",

    "Buffalo Park": "Buffalo Park",
    "Buffalo Park, East London": "Buffalo Park",

    "St George's Park": "St George's Park",
    "St George's Park, Gqeberha": "St George's Park",
    "St George's Park, Port Elizabeth": "St George's Park",

    "Harare Sports Club": "Harare Sports Club",

    "New Wanderers Stadium": "Wanderers Stadium",
    "New Wanderers Stadium, Johannesburg": "Wanderers Stadium",
    "The Wanderers Stadium": "Wanderers Stadium",
    "The Wanderers Stadium, Johannesburg": "Wanderers Stadium",

    "Boland Park": "Boland Park",
    "Boland Park, Paarl": "Boland Park",

    "Namibia Cricket Ground, Windhoek": "Namibia Cricket Ground"
}

In [11]:
matches["standardized_venue"] = matches["venue"].map(venue_mapping)

In [12]:
matches[
    matches["standardized_venue"].notna()
]["standardized_venue"].value_counts()

standardized_venue
Harare Sports Club        113
SuperSport Park            44
Queens Sports Club         37
Wanderers Stadium          36
Newlands                   26
St George's Park           26
Kingsmead                  23
Boland Park                 9
Buffalo Park                9
Mangaung Oval               8
Namibia Cricket Ground      5
Name: count, dtype: int64

In [13]:
master_data = matches[
    (matches["date"] >= "2015-01-01") &
    (matches["gender"] == "male") &
    (matches["match_type"] == "ODI") &
    (matches["standardized_venue"].notna())
].copy()

In [14]:
print("Qualifying matches:", len(master_data))

Qualifying matches: 153


In [15]:
master_data.to_csv(
    r"D:\real project DA dataset\2027 wc\master_odi_2015_12venues.csv",
    index=False
)

print("Master dataset saved successfully.")

Master dataset saved successfully.


In [16]:
print(master_data.shape)
master_data.head()

(153, 11)


,file,match_id,date,gender,match_type,venue,city,team1,team2,winner,standardized_venue
10,1007649.json,3742,2016-06-11,male,ODI,Harare Sports Club,None,Zimbabwe,India,India,Harare Sports Club
11,1007651.json,3744,2016-06-13,male,ODI,Harare Sports Club,None,Zimbabwe,India,India,Harare Sports Club
12,1007653.json,3746,2016-06-15,male,ODI,Harare Sports Club,None,Zimbabwe,India,India,Harare Sports Club
74,1059710.json,3804,2016-11-14,male,ODI,Harare Sports Club,None,Zimbabwe,Sri Lanka,Sri Lanka,Harare Sports Club
75,1059711.json,3805,2016-11-16,male,ODI,Harare Sports Club,None,Sri Lanka,West Indies,West Indies,Harare Sports Club


In [17]:
# Select the first qualifying match
sample_file = master_data.iloc[0]["file"]

sample_path = os.path.join(folder_path, sample_file)

with open(sample_path, "r", encoding="utf-8") as f:
    sample_data = json.load(f)

print(sample_data.keys())

dict_keys(['meta', 'info', 'innings'])


In [18]:
print(sample_data["info"].keys())

dict_keys(['balls_per_over', 'dates', 'event', 'gender', 'match_type', 'match_type_number', 'officials', 'outcome', 'overs', 'player_of_match', 'players', 'registry', 'season', 'team_type', 'teams', 'toss', 'venue'])


In [19]:
print(sample_data["innings"][0].keys())

dict_keys(['team', 'overs', 'powerplays'])


In [20]:
print(sample_data["innings"][0])

{'team': 'Zimbabwe', 'overs': [{'over': 0, 'deliveries': [{'actual_delivery': '0.1', 'batter': 'PJ Moor', 'bowler': 'DS Kulkarni', 'non_striker': 'CJ Chibhabha', 'runs': {'batter': 1, 'extras': 0, 'total': 1}}, {'actual_delivery': '0.2', 'batter': 'CJ Chibhabha', 'bowler': 'DS Kulkarni', 'non_striker': 'PJ Moor', 'runs': {'batter': 0, 'extras': 0, 'total': 0}}, {'actual_delivery': '0.3', 'batter': 'CJ Chibhabha', 'bowler': 'DS Kulkarni', 'non_striker': 'PJ Moor', 'runs': {'batter': 0, 'extras': 0, 'total': 0}}, {'actual_delivery': '0.4', 'batter': 'CJ Chibhabha', 'bowler': 'DS Kulkarni', 'non_striker': 'PJ Moor', 'runs': {'batter': 0, 'extras': 0, 'total': 0}}, {'actual_delivery': '0.5', 'batter': 'CJ Chibhabha', 'bowler': 'DS Kulkarni', 'non_striker': 'PJ Moor', 'runs': {'batter': 3, 'extras': 0, 'total': 3}}, {'actual_delivery': '0.6', 'batter': 'PJ Moor', 'bowler': 'DS Kulkarni', 'non_striker': 'CJ Chibhabha', 'runs': {'batter': 2, 'extras': 0, 'total': 2}}]}, {'over': 1, 'deliverie

In [23]:
# Add dismissal information to the batting records

batting_records = []

for _, match in master_data.iterrows():

    file_path = os.path.join(folder_path, match["file"])

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    info = data["info"]
    teams = info["teams"]

    for innings in data["innings"]:

        batting_team = innings["team"]
        opponent = [team for team in teams if team != batting_team][0]

        for over in innings["overs"]:

            for delivery in over["deliveries"]:

                batter = delivery["batter"]
                runs = delivery["runs"]
                extras = delivery.get("extras", {})

                is_wide = "wides" in extras

                # Check whether this batter was dismissed
                dismissed = 0

                for wicket in delivery.get("wickets", []):
                    if wicket["player_out"] == batter:
                        dismissed = 1

                batting_records.append({
                    "match_id": info["match_type_number"],
                    "date": info["dates"][0],
                    "venue": info["venue"],
                    "standardized_venue": match["standardized_venue"],
                    "batting_team": batting_team,
                    "opponent": opponent,
                    "batter": batter,
                    "runs": runs["batter"],
                    "ball_faced": 0 if is_wide else 1,
                    "four": 1 if runs["batter"] == 4 else 0,
                    "six": 1 if runs["batter"] == 6 else 0,
                    "dismissed": dismissed
                })

batting_data = pd.DataFrame(batting_records)

In [24]:
print("Batting records:", len(batting_data))

batting_data.head(10)

Batting records: 80736


,match_id,date,venue,standardized_venue,batting_team,opponent,batter,runs,ball_faced,four,six,dismissed
0,3742,2016-06-11,Harare Sports Club,Harare Sports Club,Zimbabwe,India,PJ Moor,1,1,0,0,0
1,3742,2016-06-11,Harare Sports Club,Harare Sports Club,Zimbabwe,India,CJ Chibhabha,0,1,0,0,0
2,3742,2016-06-11,Harare Sports Club,Harare Sports Club,Zimbabwe,India,CJ Chibhabha,0,1,0,0,0
3,3742,2016-06-11,Harare Sports Club,Harare Sports Club,Zimbabwe,India,CJ Chibhabha,0,1,0,0,0
4,3742,2016-06-11,Harare Sports Club,Harare Sports Club,Zimbabwe,India,CJ Chibhabha,3,1,0,0,0
5,3742,2016-06-11,Harare Sports Club,Harare Sports Club,Zimbabwe,India,PJ Moor,2,1,0,0,0
6,3742,2016-06-11,Harare Sports Club,Harare Sports Club,Zimbabwe,India,CJ Chibhabha,0,1,0,0,0
7,3742,2016-06-11,Harare Sports Club,Harare Sports Club,Zimbabwe,India,CJ Chibhabha,0,0,0,0,0
8,3742,2016-06-11,Harare Sports Club,Harare Sports Club,Zimbabwe,India,CJ Chibhabha,0,1,0,0,0
9,3742,2016-06-11,Harare Sports Club,Harare Sports Club,Zimbabwe,India,CJ Chibhabha,0,1,0,0,0


In [25]:
batting_data["dismissed"].sum()

np.int64(2097)

In [26]:
# Calculate each player's score in each innings

innings_batting = (
    batting_data
    .groupby([
        "match_id",
        "date",
        "standardized_venue",
        "batting_team",
        "opponent",
        "batter"
    ], as_index=False)
    .agg(
        runs=("runs", "sum"),
        balls_faced=("ball_faced", "sum"),
        fours=("four", "sum"),
        sixes=("six", "sum"),
        dismissed=("dismissed", "sum")
    )
)

print("Player-innings records:", len(innings_batting))
innings_batting.head()

Player-innings records: 2616


,match_id,date,standardized_venue,batting_team,opponent,batter,runs,balls_faced,fours,sixes,dismissed
0,3579,2015-01-16,Kingsmead,South Africa,West Indies,AB de Villiers,81,94,6,0,1
1,3579,2015-01-16,Kingsmead,South Africa,West Indies,DA Miller,70,68,7,2,1
2,3579,2015-01-16,Kingsmead,South Africa,West Indies,DW Steyn,6,13,1,0,0
3,3579,2015-01-16,Kingsmead,South Africa,West Indies,F Behardien,12,9,1,0,1
4,3579,2015-01-16,Kingsmead,South Africa,West Indies,F du Plessis,0,7,0,0,1


In [27]:
innings_batting["fifty"] = (
    (innings_batting["runs"] >= 50) &
    (innings_batting["runs"] < 100)
).astype(int)

innings_batting["hundred"] = (
    innings_batting["runs"] >= 100
).astype(int)

In [28]:
batting_summary = (
    innings_batting
    .groupby("batter", as_index=False)
    .agg(
        innings=("match_id", "count"),
        total_runs=("runs", "sum"),
        balls_faced=("balls_faced", "sum"),
        fours=("fours", "sum"),
        sixes=("sixes", "sum"),
        dismissals=("dismissed", "sum"),
        fifties=("fifty", "sum"),
        hundreds=("hundred", "sum")
    )
)

In [31]:
batting_summary["batting_average"] = (
    batting_summary["total_runs"] /
    batting_summary["dismissals"].replace(0, np.nan)
)

batting_summary["strike_rate"] = (
    batting_summary["total_runs"] /
    batting_summary["balls_faced"].replace(0, np.nan)
) * 100

In [32]:
top_10_batsmen = (
    batting_summary
    .sort_values("total_runs", ascending=False)
    .head(10)
)

top_10_batsmen


,batter,innings,total_runs,balls_faced,fours,sixes,dismissals,fifties,hundreds,batting_average,strike_rate
371,Q de Kock,48,2352,2269,271,51,46,11,7,51.130435,103.657999
452,Sikandar Raza,51,1851,1982,137,44,40,10,5,46.275000,93.390515
117,DA Miller,42,1690,1556,127,53,29,11,3,58.275862,108.611825
171,HM Amla,31,1565,1683,166,18,27,7,6,57.962963,92.988711
146,F du Plessis,32,1563,1623,158,17,26,8,5,60.115385,96.303142
168,HE van der Dussen,32,1298,1525,102,25,23,8,3,56.434783,85.114754
409,SC Williams,36,1256,1271,131,16,32,7,2,39.250000,98.819827
106,CR Ervine,35,1226,1466,116,20,27,7,3,45.407407,83.628922
460,T Bavuma,23,1033,1145,96,17,22,2,4,46.954545,90.218341
152,Fakhar Zaman,13,984,927,116,20,10,3,4,98.400000,106.148867


In [33]:
print("Total runs:", batting_data["runs"].sum())
print("Total fours:", batting_data["four"].sum())
print("Total sixes:", batting_data["six"].sum())
print("Total dismissals:", batting_data["dismissed"].sum())

Total runs: 67271
Total fours: 6258
Total sixes: 1268
Total dismissals: 2097


In [34]:
print(
    batting_data.groupby("match_id")["runs"]
    .sum()
    .describe()
)

count    153.000000
mean     439.679739
std      130.168598
min       46.000000
25%      362.000000
50%      462.000000
75%      533.000000
max      721.000000
Name: runs, dtype: float64


In [35]:
bowling_records = []

for _, match in master_data.iterrows():

    file_path = os.path.join(folder_path, match["file"])

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    info = data["info"]
    teams = info["teams"]

    for innings in data["innings"]:

        batting_team = innings["team"]
        bowling_team = [team for team in teams if team != batting_team][0]

        for over in innings["overs"]:

            for delivery in over["deliveries"]:

                bowler = delivery["bowler"]

                runs = delivery["runs"]
                extras = delivery.get("extras", {})

                # Legal delivery
                is_wide = "wides" in extras
                is_noball = "noballs" in extras

                legal_delivery = 0 if (is_wide or is_noball) else 1

                # Runs charged to bowler
                bowler_runs = runs["total"]

                # Remove byes and leg-byes
                bowler_runs -= extras.get("byes", 0)
                bowler_runs -= extras.get("legbyes", 0)

                # Wickets credited to bowler
                bowler_wickets = 0

                for wicket in delivery.get("wickets", []):

                    wicket_kind = wicket["kind"]

                    # These dismissals are NOT credited to the bowler
                    non_bowler_wickets = [
                        "retired hurt",
                        "retired out",
                        "obstructing the field"
                    ]

                    if wicket_kind not in non_bowler_wickets:
                        bowler_wickets += 1

                bowling_records.append({
                    "match_id": info["match_type_number"],
                    "date": info["dates"][0],
                    "venue": info["venue"],
                    "standardized_venue": match["standardized_venue"],
                    "bowling_team": bowling_team,
                    "opponent": batting_team,
                    "bowler": bowler,
                    "runs_conceded": bowler_runs,
                    "legal_delivery": legal_delivery,
                    "wicket": bowler_wickets
                })

bowling_data = pd.DataFrame(bowling_records)

In [36]:
print("Bowling records:", len(bowling_data))

bowling_data.head(10)

Bowling records: 80736


,match_id,date,venue,standardized_venue,bowling_team,opponent,bowler,runs_conceded,legal_delivery,wicket
0,3742,2016-06-11,Harare Sports Club,Harare Sports Club,India,Zimbabwe,DS Kulkarni,1,1,0
1,3742,2016-06-11,Harare Sports Club,Harare Sports Club,India,Zimbabwe,DS Kulkarni,0,1,0
2,3742,2016-06-11,Harare Sports Club,Harare Sports Club,India,Zimbabwe,DS Kulkarni,0,1,0
3,3742,2016-06-11,Harare Sports Club,Harare Sports Club,India,Zimbabwe,DS Kulkarni,0,1,0
4,3742,2016-06-11,Harare Sports Club,Harare Sports Club,India,Zimbabwe,DS Kulkarni,3,1,0
5,3742,2016-06-11,Harare Sports Club,Harare Sports Club,India,Zimbabwe,DS Kulkarni,2,1,0
6,3742,2016-06-11,Harare Sports Club,Harare Sports Club,India,Zimbabwe,BB Sran,0,1,0
7,3742,2016-06-11,Harare Sports Club,Harare Sports Club,India,Zimbabwe,BB Sran,1,0,0
8,3742,2016-06-11,Harare Sports Club,Harare Sports Club,India,Zimbabwe,BB Sran,0,1,0
9,3742,2016-06-11,Harare Sports Club,Harare Sports Club,India,Zimbabwe,BB Sran,0,1,0


In [37]:
print("Total wickets:", bowling_data["wicket"].sum())
print("Total runs conceded:", bowling_data["runs_conceded"].sum())

Total wickets: 2165
Total runs conceded: 69857


In [38]:
bowling_match = (
    bowling_data
    .groupby([
        "match_id",
        "date",
        "standardized_venue",
        "bowling_team",
        "opponent",
        "bowler"
    ], as_index=False)
    .agg(
        runs_conceded=("runs_conceded", "sum"),
        legal_deliveries=("legal_delivery", "sum"),
        wickets=("wicket", "sum")
    )
)

In [40]:
bowling_summary = (
    bowling_data
    .groupby("bowler", as_index=False)
    .agg(
        matches=("match_id", "nunique"),
        legal_deliveries=("legal_delivery", "sum"),
        runs_conceded=("runs_conceded", "sum"),
        wickets=("wicket", "sum")
    )
)

In [41]:
bowling_summary["overs"] = (
    bowling_summary["legal_deliveries"] // 6
    + (bowling_summary["legal_deliveries"] % 6) / 10
)

In [42]:
bowling_summary["economy"] = (
    bowling_summary["runs_conceded"] /
    (bowling_summary["legal_deliveries"] / 6)
)

In [43]:
bowling_summary["bowling_average"] = (
    bowling_summary["runs_conceded"] /
    bowling_summary["wickets"].replace(0, np.nan)
)

In [44]:
bowling_summary["bowling_strike_rate"] = (
    bowling_summary["legal_deliveries"] /
    bowling_summary["wickets"].replace(0, np.nan)
)

In [45]:
top_10_bowlers = (
    bowling_summary
    .sort_values(
        ["wickets", "bowling_average"],
        ascending=[False, True]
    )
    .head(10)
)

top_10_bowlers

,bowler,matches,legal_deliveries,runs_conceded,wickets,overs,economy,bowling_average,bowling_strike_rate
171,K Rabada,43,2316,2017,71,386.0,5.225389,28.408451,32.619718
140,Imran Tahir,34,1843,1433,64,307.1,4.665220,22.390625,28.796875
15,AL Phehlukwayo,42,1694,1592,55,282.2,5.638725,28.945455,30.800000
193,L Ngidi,28,1346,1272,45,224.2,5.670134,28.266667,29.911111
335,Sikandar Raza,46,1890,1524,44,315.0,4.838095,34.636364,42.954545
286,R Ngarava,24,1219,1058,39,203.1,5.207547,27.128205,31.256410
347,T Shamsi,26,1317,1259,32,219.3,5.735763,39.343750,41.156250
161,JO Holder,16,851,742,30,141.5,5.231492,24.733333,28.366667
41,B Muzarabani,27,1342,1107,30,223.4,4.949329,36.900000,44.733333
354,TL Chatara,22,1011,959,27,168.3,5.691395,35.518519,37.444444


In [46]:
team_records = []

for _, match in master_data.iterrows():

    team1 = match["team1"]
    team2 = match["team2"]
    winner = match["winner"]

    # Team 1
    team_records.append({
        "match_id": match["match_id"],
        "date": match["date"],
        "venue": match["standardized_venue"],
        "team": team1,
        "opponent": team2,
        "result": (
            "Win" if winner == team1
            else "Loss" if winner == team2
            else "No Result"
        )
    })

    # Team 2
    team_records.append({
        "match_id": match["match_id"],
        "date": match["date"],
        "venue": match["standardized_venue"],
        "team": team2,
        "opponent": team1,
        "result": (
            "Win" if winner == team2
            else "Loss" if winner == team1
            else "No Result"
        )
    })

team_matches = pd.DataFrame(team_records)

print("Team-match records:", len(team_matches))
team_matches.head()

Team-match records: 306


,match_id,date,venue,team,opponent,result
0,3742,2016-06-11,Harare Sports Club,Zimbabwe,India,Loss
1,3742,2016-06-11,Harare Sports Club,India,Zimbabwe,Win
2,3744,2016-06-13,Harare Sports Club,Zimbabwe,India,Loss
3,3744,2016-06-13,Harare Sports Club,India,Zimbabwe,Win
4,3746,2016-06-15,Harare Sports Club,Zimbabwe,India,Loss


In [47]:
team_summary = (
    team_matches
    .groupby("team", as_index=False)
    .agg(
        matches=("match_id", "nunique"),
        wins=("result", lambda x: (x == "Win").sum()),
        losses=("result", lambda x: (x == "Loss").sum()),
        no_results=("result", lambda x: (x == "No Result").sum())
    )
)

In [48]:
team_summary["win_percentage"] = (
    team_summary["wins"] /
    (team_summary["wins"] + team_summary["losses"])
) * 100

In [49]:
team_summary = team_summary.sort_values(
    "win_percentage",
    ascending=False
)

team_summary

,team,matches,wins,losses,no_results,win_percentage
4,India,21,16,5,0,76.190476
11,Pakistan,19,14,5,0,73.684211
14,South Africa,71,46,23,2,66.666667
1,Bangladesh,14,7,7,0,50.000000
5,Ireland,16,7,7,2,50.000000
13,Scotland,8,3,3,2,50.000000
18,West Indies,18,8,9,1,47.058824
10,Oman,5,2,3,0,40.000000
9,New Zealand,5,2,3,0,40.000000
15,Sri Lanka,19,7,11,1,38.888889


In [50]:
team_runs_scored = (
    batting_data
    .groupby("batting_team", as_index=False)
    .agg(
        runs_scored=("runs", "sum")
    )
)

team_runs_scored.columns = [
    "team",
    "runs_scored"
]

In [51]:
team_runs_conceded = (
    bowling_data
    .groupby("bowling_team", as_index=False)
    .agg(
        runs_conceded=("runs_conceded", "sum")
    )

)

team_runs_conceded.columns = [
    "team",
    "runs_conceded"
]

In [52]:
team_summary = (
    team_summary
    .merge(team_runs_scored, on="team", how="left")
    .merge(team_runs_conceded, on="team", how="left")
)

In [53]:
team_summary["avg_runs_scored"] = (
    team_summary["runs_scored"] /
    team_summary["matches"]
)

team_summary["avg_runs_conceded"] = (
    team_summary["runs_conceded"] /
    team_summary["matches"]
)

In [54]:
team_wickets_taken = (
    bowling_data
    .groupby("bowling_team", as_index=False)
    .agg(
        wickets_taken=("wicket", "sum")
    )

)

team_wickets_taken.columns = [
    "team",
    "wickets_taken"
]

In [55]:
team_wickets_lost = (
    batting_data
    .groupby("batting_team", as_index=False)
    .agg(
        wickets_lost=("dismissed", "sum")
    )
)

team_wickets_lost.columns = [
    "team",
    "wickets_lost"
]

In [56]:
team_summary = (
    team_summary
    .merge(team_wickets_taken, on="team", how="left")
    .merge(team_wickets_lost, on="team", how="left")
)

In [57]:
team_summary["run_rate_scored"] = (
    team_summary["runs_scored"] /
    team_summary["matches"]
)

team_summary["run_rate_conceded"] = (
    team_summary["runs_conceded"] /
    team_summary["matches"]
)

team_summary["run_differential"] = (
    team_summary["run_rate_scored"] -
    team_summary["run_rate_conceded"]
)

In [58]:
team_summary = team_summary[
    [
        "team",
        "matches",
        "wins",
        "losses",
        "no_results",
        "win_percentage",
        "runs_scored",
        "runs_conceded",
        "avg_runs_scored",
        "avg_runs_conceded",
        "wickets_taken",
        "wickets_lost",
        "run_differential"
    ]
]

team_summary = team_summary.sort_values(
    "win_percentage",
    ascending=False
)

team_summary

,team,matches,wins,losses,no_results,win_percentage,runs_scored,runs_conceded,avg_runs_scored,avg_runs_conceded,wickets_taken,wickets_lost,run_differential
0,India,21,16,5,0,76.190476,4514,4213,214.952381,200.619048,178,103,14.333333
1,Pakistan,19,14,5,0,73.684211,4763,3997,250.684211,210.368421,143,107,40.315789
2,South Africa,71,46,23,2,66.666667,17739,16715,249.845070,235.422535,559,446,14.422535
3,Bangladesh,14,7,7,0,50.000000,3079,3287,219.928571,234.785714,109,97,-14.857143
4,Ireland,16,7,7,2,50.000000,2986,3342,186.625000,208.875000,120,96,-22.250000
5,Scotland,8,3,3,2,50.000000,1369,1621,171.125000,202.625000,74,47,-31.500000
6,West Indies,18,8,9,1,47.058824,4060,4430,225.555556,246.111111,128,141,-20.555556
7,Oman,5,2,3,0,40.000000,1042,1208,208.400000,241.600000,35,36,-33.200000
8,New Zealand,5,2,3,0,40.000000,1246,1338,249.200000,267.600000,36,29,-18.400000
9,Sri Lanka,19,7,11,1,38.888889,3730,4320,196.315789,227.368421,121,133,-31.052632


In [59]:
team_venue = (
    team_matches
    .groupby(["team", "venue"], as_index=False)
    .agg(
        matches=("match_id", "nunique"),
        wins=("result", lambda x: (x == "Win").sum()),
        losses=("result", lambda x: (x == "Loss").sum()),
        no_results=("result", lambda x: (x == "No Result").sum())
    )
)

team_venue["win_percentage"] = (
    team_venue["wins"] /
    (team_venue["wins"] + team_venue["losses"])
) * 100

team_venue = team_venue.sort_values(
    ["venue", "win_percentage"],
    ascending=[True, False]
)

team_venue

,team,venue,matches,wins,losses,no_results,win_percentage
37,Pakistan,Boland Park,1,1,0,0,100.000000
49,South Africa,Boland Park,7,5,2,0,71.428571
19,India,Boland Park,3,1,2,0,33.333333
0,Australia,Boland Park,1,0,1,0,0.000000
7,Bangladesh,Boland Park,1,0,1,0,0.000000
...,...,...,...,...,...,...,...
6,Australia,Wanderers Stadium,2,0,2,0,0.000000
11,Bangladesh,Wanderers Stadium,1,0,1,0,0.000000
31,Netherlands,Wanderers Stadium,1,0,1,0,0.000000
63,Sri Lanka,Wanderers Stadium,2,0,2,0,0.000000


In [60]:
team_venue_pivot = team_venue.pivot(
    index="team",
    columns="venue",
    values="win_percentage"
)

team_venue_pivot

venue,Boland Park,Buffalo Park,Harare Sports Club,Kingsmead,Mangaung Oval,Namibia Cricket Ground,Newlands,Queens Sports Club,St George's Park,SuperSport Park,Wanderers Stadium
team,,,,,,,,,,,
Australia,0.000000,NaN,NaN,0.000000,66.666667,NaN,0.000000,NaN,0.0,0.000000,0.000000
Bangladesh,0.000000,0.000000,55.555556,NaN,NaN,NaN,NaN,NaN,NaN,100.000000,0.000000
England,NaN,NaN,NaN,NaN,33.333333,NaN,0.000000,NaN,100.0,0.000000,50.000000
Hong Kong,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
India,33.333333,NaN,100.000000,100.000000,NaN,NaN,50.000000,NaN,50.0,100.000000,50.000000
Ireland,NaN,NaN,50.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Namibia,NaN,NaN,NaN,NaN,NaN,33.333333,NaN,NaN,NaN,NaN,NaN
Nepal,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Netherlands,NaN,NaN,33.333333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000


In [61]:
team_venue_stats = (
    team_matches
    .groupby(["team", "venue"], as_index=False)
    .agg(
        matches=("match_id", "nunique"),
        wins=("result", lambda x: (x == "Win").sum()),
        losses=("result", lambda x: (x == "Loss").sum())
    )
)

team_venue_stats["win_percentage"] = (
    team_venue_stats["wins"] /
    (team_venue_stats["wins"] + team_venue_stats["losses"])
) * 100

team_venue_stats

,team,venue,matches,wins,losses,win_percentage
0,Australia,Boland Park,1,0,1,0.000000
1,Australia,Kingsmead,1,0,1,0.000000
2,Australia,Mangaung Oval,3,2,1,66.666667
3,Australia,Newlands,1,0,1,0.000000
4,Australia,St George's Park,1,0,1,0.000000
...,...,...,...,...,...,...
73,West Indies,Wanderers Stadium,1,0,1,0.000000
74,Zimbabwe,Boland Park,1,0,1,0.000000
75,Zimbabwe,Harare Sports Club,52,21,29,42.000000
76,Zimbabwe,Mangaung Oval,1,0,1,0.000000


In [62]:
overall_win = (
    team_matches
    .groupby("team")
    .agg(
        wins=("result", lambda x: (x == "Win").sum()),
        losses=("result", lambda x: (x == "Loss").sum())
    )
    .reset_index()
)

overall_win["overall_win_percentage"] = (
    overall_win["wins"] /
    (overall_win["wins"] + overall_win["losses"])
) * 100

In [63]:
team_venue_stats = team_venue_stats.merge(
    overall_win[["team", "overall_win_percentage"]],
    on="team",
    how="left"
)

In [64]:
k = 5

team_venue_stats["adjusted_venue_win_percentage"] = (
    (
        team_venue_stats["matches"] *
        team_venue_stats["win_percentage"]
    )
    +
    (
        k *
        team_venue_stats["overall_win_percentage"]
    )
) / (
    team_venue_stats["matches"] + k
)

In [65]:
print(sorted(master_data["standardized_venue"].unique()))
print("Number of venues:", master_data["standardized_venue"].nunique())

['Boland Park', 'Buffalo Park', 'Harare Sports Club', 'Kingsmead', 'Mangaung Oval', 'Namibia Cricket Ground', 'Newlands', 'Queens Sports Club', "St George's Park", 'SuperSport Park', 'Wanderers Stadium']
Number of venues: 11


In [66]:
# Show every standardized venue and number of qualifying matches

venue_counts = (
    master_data
    .groupby("standardized_venue")
    .size()
    .sort_values(ascending=False)
)

venue_counts

standardized_venue
Harare Sports Club        64
SuperSport Park           15
Wanderers Stadium         14
Queens Sports Club        13
Newlands                   9
Kingsmead                  8
St George's Park           8
Boland Park                7
Mangaung Oval              7
Namibia Cricket Ground     5
Buffalo Park               3
dtype: int64

In [67]:
# All South Africa, Zimbabwe and Namibia venue/city combinations
# from the complete 2,565-match dataset

host_country_venues = (
    matches[
        matches["city"].isin([
            "Bloemfontein",
            "Bulawayo",
            "Cape Town",
            "Centurion",
            "Durban",
            "East London",
            "Gqeberha",
            "Harare",
            "Johannesburg",
            "Paarl",
            "Port Elizabeth",
            "Windhoek"
        ])
    ][["venue", "city"]]
    .drop_duplicates()
    .sort_values(["city", "venue"])
)

host_country_venues

,venue,city
1805,Chevrolet Park,Bloemfontein
1286,Goodyear Park,Bloemfontein
2123,"Goodyear Park, Bloemfontein",Bloemfontein
208,Mangaung Oval,Bloemfontein
613,"Mangaung Oval, Bloemfontein",Bloemfontein
1497,OUTsurance Oval,Bloemfontein
722,Bulawayo Athletic Club,Bulawayo
76,Queens Sports Club,Bulawayo
721,"Queens Sports Club, Bulawayo",Bulawayo
155,Newlands,Cape Town


In [68]:
official_venues = pd.DataFrame({
    "venue": [
        "Wanderers Stadium",
        "SuperSport Park",
        "Newlands",
        "Kingsmead",
        "St George's Park",
        "Mangaung Oval",
        "Boland Park",
        "Buffalo Park",
        "Harare Sports Club",
        "Queens Sports Club",
        "Mosi-oa-Tunya International Cricket Stadium",
        "Namibia Cricket Ground"
    ],
    
    "city": [
        "Johannesburg",
        "Tshwane",
        "Cape Town",
        "Durban",
        "Gqeberha",
        "Bloemfontein",
        "Paarl",
        "East London",
        "Harare",
        "Bulawayo",
        "Victoria Falls",
        "Windhoek"
    ],
    
    "country": [
        "South Africa",
        "South Africa",
        "South Africa",
        "South Africa",
        "South Africa",
        "South Africa",
        "South Africa",
        "South Africa",
        "Zimbabwe",
        "Zimbabwe",
        "Zimbabwe",
        "Namibia"
    ]
})

official_venues

,venue,city,country
0,Wanderers Stadium,Johannesburg,South Africa
1,SuperSport Park,Tshwane,South Africa
2,Newlands,Cape Town,South Africa
3,Kingsmead,Durban,South Africa
4,St George's Park,Gqeberha,South Africa
5,Mangaung Oval,Bloemfontein,South Africa
6,Boland Park,Paarl,South Africa
7,Buffalo Park,East London,South Africa
8,Harare Sports Club,Harare,Zimbabwe
9,Queens Sports Club,Bulawayo,Zimbabwe


In [69]:
print("Official venues:", len(official_venues))

Official venues: 12


In [70]:
venue_batting = (
    batting_data
    .groupby(
        ["standardized_venue", "batter"],
        as_index=False
    )
    .agg(
        innings=("match_id", "nunique"),
        runs=("runs", "sum"),
        balls_faced=("ball_faced", "sum"),
        fours=("four", "sum"),
        sixes=("six", "sum"),
        dismissals=("dismissed", "sum")
    )
)

In [71]:
venue_batting["batting_average"] = (
    venue_batting["runs"] /
    venue_batting["dismissals"].replace(0, np.nan)
)

venue_batting["strike_rate"] = (
    venue_batting["runs"] /
    venue_batting["balls_faced"].replace(0, np.nan)
) * 100

In [73]:
top_10_venue_batsmen = (
    venue_batting
    .sort_values(
        ["standardized_venue", "runs"],
        ascending=[True, False]
    )
    .groupby("standardized_venue")
    .head(10)
)

top_10_venue_batsmen

,standardized_venue,batter,innings,runs,balls_faced,fours,sixes,dismissals,batting_average,strike_rate
28,Boland Park,H Klaasen,4,289,300,23,6,3,96.333333,96.333333
2,Boland Park,AB de Villiers,1,176,104,15,7,1,176.000000,169.230769
30,Boland Park,HE van der Dussen,4,176,165,12,4,2,88.000000,106.666667
81,Boland Park,T Bavuma,3,171,209,14,0,3,57.000000,81.818182
62,Boland Park,Q de Kock,4,166,184,11,3,4,41.500000,90.217391
...,...,...,...,...,...,...,...,...,...,...
1062,Wanderers Stadium,AB de Villiers,4,271,150,19,19,2,135.500000,180.666667
1106,Wanderers Stadium,Fakhar Zaman,2,237,199,25,10,2,118.500000,119.095477
1110,Wanderers Stadium,HE van der Dussen,9,213,274,19,6,7,30.428571,77.737226
1172,Wanderers Stadium,RR Rossouw,2,203,196,21,2,2,101.500000,103.571429


In [74]:
venue_bowling = (
    bowling_data
    .groupby(
        ["standardized_venue", "bowler"],
        as_index=False
    )
    .agg(
        matches=("match_id", "nunique"),
        legal_deliveries=("legal_delivery", "sum"),
        runs_conceded=("runs_conceded", "sum"),
        wickets=("wicket", "sum")
    )
)

In [75]:
venue_bowling["overs"] = (
    venue_bowling["legal_deliveries"] // 6
    + (venue_bowling["legal_deliveries"] % 6) / 10
)

In [76]:
venue_bowling["economy"] = (
    venue_bowling["runs_conceded"] /
    (venue_bowling["legal_deliveries"] / 6)
)

In [77]:
venue_bowling["bowling_average"] = (
    venue_bowling["runs_conceded"] /
    venue_bowling["wickets"].replace(0, np.nan)
)

In [78]:
venue_bowling["bowling_strike_rate"] = (
    venue_bowling["legal_deliveries"] /
    venue_bowling["wickets"].replace(0, np.nan)
)

In [79]:
top_10_venue_bowlers = (
    venue_bowling
    .sort_values(
        ["standardized_venue", "wickets"],
        ascending=[True, False]
    )
    .groupby("standardized_venue")
    .head(10)
)

top_10_venue_bowlers

,standardized_venue,bowler,matches,legal_deliveries,runs_conceded,wickets,overs,economy,bowling_average,bowling_strike_rate
3,Boland Park,AL Phehlukwayo,6,252,211,11,42.0,5.023810,19.181818,22.909091
50,Boland Park,T Shamsi,5,294,280,7,49.0,5.714286,40.000000,42.000000
18,Boland Park,Imran Tahir,2,120,94,5,20.0,4.700000,18.800000,24.000000
22,Boland Park,K Rabada,3,174,120,5,29.0,4.137931,24.000000,34.800000
26,Boland Park,L Ngidi,3,156,129,5,26.0,4.961538,25.800000,31.200000
...,...,...,...,...,...,...,...,...,...,...
757,Wanderers Stadium,AU Rashid,2,92,89,6,15.2,5.804348,14.833333,15.333333
760,Wanderers Stadium,Arshdeep Singh,1,60,37,5,10.0,3.700000,7.400000,12.000000
834,Wanderers Stadium,SSB Magala,1,54,43,5,9.0,4.777778,8.600000,10.800000
837,Wanderers Stadium,Shaheen Shah Afridi,3,150,169,5,25.0,6.760000,33.800000,30.000000


In [80]:
team_venue_batting = (
    batting_data
    .groupby(
        ["standardized_venue", "batting_team"],
        as_index=False
    )
    .agg(
        runs=("runs", "sum"),
        balls_faced=("ball_faced", "sum"),
        fours=("four", "sum"),
        sixes=("six", "sum"),
        dismissals=("dismissed", "sum")
    )
)

In [81]:
team_venue_batting["batting_average"] = (
    team_venue_batting["runs"] /
    team_venue_batting["dismissals"].replace(0, np.nan)
)

team_venue_batting["strike_rate"] = (
    team_venue_batting["runs"] /
    team_venue_batting["balls_faced"].replace(0, np.nan)
) * 100

In [84]:
print("Rows:", len(team_venue_batting))
print("Teams:", team_venue_batting["batting_team"].nunique())
print("Venues:", team_venue_batting["standardized_venue"].nunique())

Rows: 77
Teams: 20
Venues: 11


In [82]:
team_venue_bowling = (
    bowling_data
    .groupby(
        ["standardized_venue", "bowling_team"],
        as_index=False
    )
    .agg(
        legal_deliveries=("legal_delivery", "sum"),
        runs_conceded=("runs_conceded", "sum"),
        wickets=("wicket", "sum")
    )
)

In [85]:
team_venue_bowling["economy"] = (
    team_venue_bowling["runs_conceded"] /
    (team_venue_bowling["legal_deliveries"] / 6)
)

team_venue_bowling["bowling_average"] = (
    team_venue_bowling["runs_conceded"] /
    team_venue_bowling["wickets"].replace(0, np.nan)
)

team_venue_bowling["bowling_strike_rate"] = (
    team_venue_bowling["legal_deliveries"] /
    team_venue_bowling["wickets"].replace(0, np.nan)
)
team_venue_bowling.head(20)

,standardized_venue,bowling_team,legal_deliveries,runs_conceded,wickets,economy,bowling_average,bowling_strike_rate
0,Boland Park,Australia,300,285,7,5.700000,40.714286,42.857143
1,Boland Park,Bangladesh,300,351,6,7.020000,58.500000,50.000000
2,Boland Park,India,864,790,17,5.486111,46.470588,50.823529
3,Boland Park,Pakistan,300,235,9,4.700000,26.111111,33.333333
4,Boland Park,South Africa,2052,1764,59,5.157895,29.898305,34.779661
5,Boland Park,Zimbabwe,275,231,6,5.040000,38.500000,45.833333
6,Buffalo Park,Bangladesh,300,366,6,7.320000,61.000000,50.000000
7,Buffalo Park,South Africa,746,614,28,4.938338,21.928571,26.642857
8,Buffalo Park,West Indies,398,405,11,6.105528,36.818182,36.181818
9,Harare Sports Club,Bangladesh,2348,1957,74,5.000852,26.445946,31.729730


In [86]:
print("Rows:", len(team_venue_bowling))
print("Teams:", team_venue_bowling["bowling_team"].nunique())
print("Venues:", team_venue_bowling["standardized_venue"].nunique())

Rows: 78
Teams: 20
Venues: 11


In [87]:
print(sorted(team_venue_batting["batting_team"].unique()))

['Australia', 'Bangladesh', 'England', 'Hong Kong', 'India', 'Ireland', 'Namibia', 'Nepal', 'Netherlands', 'New Zealand', 'Oman', 'Pakistan', 'Papua New Guinea', 'Scotland', 'South Africa', 'Sri Lanka', 'United Arab Emirates', 'United States of America', 'West Indies', 'Zimbabwe']


In [88]:
print(sorted(team_venue_bowling["bowling_team"].unique()))

['Australia', 'Bangladesh', 'England', 'Hong Kong', 'India', 'Ireland', 'Namibia', 'Nepal', 'Netherlands', 'New Zealand', 'Oman', 'Pakistan', 'Papua New Guinea', 'Scotland', 'South Africa', 'Sri Lanka', 'United Arab Emirates', 'United States of America', 'West Indies', 'Zimbabwe']


In [89]:
wc2027_teams = [
    "Australia",
    "Bangladesh",
    "England",
    "India",
    "Ireland",
    "New Zealand",
    "Pakistan",
    "Scotland",
    "South Africa",
    "Sri Lanka",
    "West Indies",
    "Zimbabwe",
    "Namibia",
    "Afghanistan"
]

print("2027 World Cup teams:", len(wc2027_teams))
print(wc2027_teams)

2027 World Cup teams: 14
['Australia', 'Bangladesh', 'England', 'India', 'Ireland', 'New Zealand', 'Pakistan', 'Scotland', 'South Africa', 'Sri Lanka', 'West Indies', 'Zimbabwe', 'Namibia', 'Afghanistan']


In [90]:
wc2027_batting = team_venue_batting[
    team_venue_batting["batting_team"].isin(wc2027_teams)
].copy()

print("Rows:", len(wc2027_batting))
print("Teams:", wc2027_batting["batting_team"].nunique())

Rows: 66
Teams: 13


In [91]:
wc2027_bowling = team_venue_bowling[
    team_venue_bowling["bowling_team"].isin(wc2027_teams)
].copy()

print("Rows:", len(wc2027_bowling))
print("Teams:", wc2027_bowling["bowling_team"].nunique())

Rows: 67
Teams: 13


In [92]:
missing_teams = set(wc2027_teams) - set(wc2027_batting["batting_team"])

print("Teams without historical venue batting data:")
print(sorted(missing_teams))

Teams without historical venue batting data:
['Afghanistan']


In [93]:
print("2027 teams:", len(wc2027_teams))
print("Teams with historical data:", wc2027_batting["batting_team"].nunique())

2027 teams: 14
Teams with historical data: 13


In [95]:
team_venue_batting["innings"] = (
    batting_data
    .groupby(["standardized_venue", "batting_team", "match_id"])
    .size()
    .reset_index()
    .groupby(["standardized_venue", "batting_team"])
    .size()
    .values
)

In [101]:
team_venue_batting.head(20)

,standardized_venue,batting_team,runs,balls_faced,fours,sixes,dismissals,batting_average,strike_rate,runs_per_match,innings,runs_per_innings
0,Boland Park,Australia,210,271,13,0,10,21.000000,77.490775,NaN,1,210.000000
1,Boland Park,Bangladesh,236,287,21,3,10,23.600000,82.229965,NaN,1,236.000000
2,Boland Park,India,793,903,66,14,22,36.045455,87.818383,NaN,3,264.333333
3,Boland Park,Pakistan,229,297,17,5,7,32.714286,77.104377,NaN,1,229.000000
4,Boland Park,South Africa,1838,2045,142,28,45,40.844444,89.877751,NaN,7,262.571429
5,Boland Park,Zimbabwe,218,297,28,1,10,21.800000,73.400673,NaN,1,218.000000
6,Buffalo Park,Bangladesh,161,245,21,0,10,16.100000,65.714286,NaN,1,161.000000
7,Buffalo Park,South Africa,758,699,76,18,17,44.588235,108.440629,NaN,3,252.666667
8,Buffalo Park,West Indies,427,504,36,15,18,23.722222,84.722222,NaN,2,213.500000
9,Harare Sports Club,Bangladesh,2057,2493,190,30,60,34.283333,82.511031,NaN,9,228.555556


In [100]:
team_venue_batting.describe()

,runs,balls_faced,fours,sixes,dismissals,batting_average,strike_rate,runs_per_match,innings,runs_per_innings
count,77.000000,77.000000,77.000000,77.000000,77.000000,76.000000,77.000000,0.0,77.000000,77.000000
mean,873.649351,1022.337662,81.272727,16.467532,27.233766,33.614999,85.147361,NaN,3.909091,220.464985
std,1384.679288,1682.182633,128.328488,25.560344,46.622475,16.598074,12.850661,NaN,6.454663,52.897027
min,11.000000,12.000000,2.000000,0.000000,0.000000,7.200000,50.000000,NaN,1.000000,11.000000
25%,252.000000,296.000000,24.000000,5.000000,10.000000,22.550000,77.490775,NaN,1.000000,195.444444
50%,457.000000,527.000000,44.000000,9.000000,16.000000,30.802083,86.597938,NaN,2.000000,228.000000
75%,843.000000,903.000000,81.000000,17.000000,23.000000,39.595722,92.775468,NaN,3.000000,253.125000
max,10760.000000,13441.000000,992.000000,167.000000,383.000000,102.666667,120.000000,NaN,52.000000,360.000000


In [98]:
team_venue_batting["runs_per_innings"] = (
    team_venue_batting["runs"] /
    team_venue_batting["innings"]
)

In [99]:
team_venue_batting[
    [
        "standardized_venue",
        "batting_team",
        "runs",
        "innings",
        "runs_per_innings",
        "batting_average",
        "strike_rate"
    ]
].head(20)

,standardized_venue,batting_team,runs,innings,runs_per_innings,batting_average,strike_rate
0,Boland Park,Australia,210,1,210.000000,21.000000,77.490775
1,Boland Park,Bangladesh,236,1,236.000000,23.600000,82.229965
2,Boland Park,India,793,3,264.333333,36.045455,87.818383
3,Boland Park,Pakistan,229,1,229.000000,32.714286,77.104377
4,Boland Park,South Africa,1838,7,262.571429,40.844444,89.877751
5,Boland Park,Zimbabwe,218,1,218.000000,21.800000,73.400673
6,Buffalo Park,Bangladesh,161,1,161.000000,16.100000,65.714286
7,Buffalo Park,South Africa,758,3,252.666667,44.588235,108.440629
8,Buffalo Park,West Indies,427,2,213.500000,23.722222,84.722222
9,Harare Sports Club,Bangladesh,2057,9,228.555556,34.283333,82.511031


In [102]:
batting_strength = team_venue_batting[
    [
        "standardized_venue",
        "batting_team",
        "runs",
        "innings",
        "runs_per_innings",
        "batting_average",
        "strike_rate"
    ]
].copy()

batting_strength.head(20)

,standardized_venue,batting_team,runs,innings,runs_per_innings,batting_average,strike_rate
0,Boland Park,Australia,210,1,210.000000,21.000000,77.490775
1,Boland Park,Bangladesh,236,1,236.000000,23.600000,82.229965
2,Boland Park,India,793,3,264.333333,36.045455,87.818383
3,Boland Park,Pakistan,229,1,229.000000,32.714286,77.104377
4,Boland Park,South Africa,1838,7,262.571429,40.844444,89.877751
5,Boland Park,Zimbabwe,218,1,218.000000,21.800000,73.400673
6,Buffalo Park,Bangladesh,161,1,161.000000,16.100000,65.714286
7,Buffalo Park,South Africa,758,3,252.666667,44.588235,108.440629
8,Buffalo Park,West Indies,427,2,213.500000,23.722222,84.722222
9,Harare Sports Club,Bangladesh,2057,9,228.555556,34.283333,82.511031


In [104]:
print("Rows:", len(batting_strength))
print("Venues:", batting_strength["standardized_venue"].nunique())
print("Teams:", batting_strength["batting_team"].nunique())

Rows: 77
Venues: 11
Teams: 20


In [105]:
bowling_strength = team_venue_bowling[
    [
        "standardized_venue",
        "bowling_team",
        "legal_deliveries",
        "runs_conceded",
        "wickets",
        "economy",
        "bowling_average",
        "bowling_strike_rate"
    ]
].copy()

bowling_strength.head(20)

,standardized_venue,bowling_team,legal_deliveries,runs_conceded,wickets,economy,bowling_average,bowling_strike_rate
0,Boland Park,Australia,300,285,7,5.700000,40.714286,42.857143
1,Boland Park,Bangladesh,300,351,6,7.020000,58.500000,50.000000
2,Boland Park,India,864,790,17,5.486111,46.470588,50.823529
3,Boland Park,Pakistan,300,235,9,4.700000,26.111111,33.333333
4,Boland Park,South Africa,2052,1764,59,5.157895,29.898305,34.779661
5,Boland Park,Zimbabwe,275,231,6,5.040000,38.500000,45.833333
6,Buffalo Park,Bangladesh,300,366,6,7.320000,61.000000,50.000000
7,Buffalo Park,South Africa,746,614,28,4.938338,21.928571,26.642857
8,Buffalo Park,West Indies,398,405,11,6.105528,36.818182,36.181818
9,Harare Sports Club,Bangladesh,2348,1957,74,5.000852,26.445946,31.729730


In [106]:
print("Rows:", len(bowling_strength))
print("Venues:", bowling_strength["standardized_venue"].nunique())
print("Teams:", bowling_strength["bowling_team"].nunique())

Rows: 78
Venues: 11
Teams: 20


In [107]:
bowling_strength.describe()

,legal_deliveries,runs_conceded,wickets,economy,bowling_average,bowling_strike_rate
count,78.000000,78.000000,78.000000,78.000000,78.000000,78.000000
mean,1006.987179,895.602564,27.756410,5.622208,42.569533,43.791963
std,1640.483002,1400.286977,42.117395,1.003987,28.333243,22.011846
min,68.000000,69.000000,2.000000,3.655400,16.100000,21.200000
25%,300.000000,317.500000,7.000000,4.962850,27.622003,31.175795
50%,544.500000,496.000000,15.000000,5.394086,35.141649,37.500000
75%,861.750000,763.000000,26.000000,6.101205,46.321691,49.833333
max,13248.000000,11290.000000,314.000000,8.700000,217.500000,150.000000


In [108]:
bowling_matches = (
    bowling_data
    .groupby(["standardized_venue", "bowling_team"])["match_id"]
    .nunique()
    .reset_index(name="matches")
)

bowling_matches.head(20)

,standardized_venue,bowling_team,matches
0,Boland Park,Australia,1
1,Boland Park,Bangladesh,1
2,Boland Park,India,3
3,Boland Park,Pakistan,1
4,Boland Park,South Africa,7
5,Boland Park,Zimbabwe,1
6,Buffalo Park,Bangladesh,1
7,Buffalo Park,South Africa,3
8,Buffalo Park,West Indies,2
9,Harare Sports Club,Bangladesh,9


In [109]:
bowling_strength = bowling_strength.merge(
    bowling_matches,
    on=["standardized_venue", "bowling_team"],
    how="left"
)

In [110]:
bowling_strength["wickets_per_match"] = (
    bowling_strength["wickets"] /
    bowling_strength["matches"]
)

bowling_strength["runs_conceded_per_match"] = (
    bowling_strength["runs_conceded"] /
    bowling_strength["matches"]
)

In [111]:
bowling_strength[
    [
        "standardized_venue",
        "bowling_team",
        "matches",
        "wickets",
        "wickets_per_match",
        "economy",
        "bowling_average",
        "bowling_strike_rate"
    ]
].head(20)

,standardized_venue,bowling_team,matches,wickets,wickets_per_match,economy,bowling_average,bowling_strike_rate
0,Boland Park,Australia,1,7,7.000000,5.700000,40.714286,42.857143
1,Boland Park,Bangladesh,1,6,6.000000,7.020000,58.500000,50.000000
2,Boland Park,India,3,17,5.666667,5.486111,46.470588,50.823529
3,Boland Park,Pakistan,1,9,9.000000,4.700000,26.111111,33.333333
4,Boland Park,South Africa,7,59,8.428571,5.157895,29.898305,34.779661
5,Boland Park,Zimbabwe,1,6,6.000000,5.040000,38.500000,45.833333
6,Buffalo Park,Bangladesh,1,6,6.000000,7.320000,61.000000,50.000000
7,Buffalo Park,South Africa,3,28,9.333333,4.938338,21.928571,26.642857
8,Buffalo Park,West Indies,2,11,5.500000,6.105528,36.818182,36.181818
9,Harare Sports Club,Bangladesh,9,74,8.222222,5.000852,26.445946,31.729730


In [114]:
%whos

Variable               Type             Data/Info
-------------------------------------------------
batter                 str              DAS Gunaratne
batting_data           DataFrame        Shape: (80736, 12)
batting_records        list             n=80736
batting_strength       DataFrame        Shape: (77, 7)
batting_summary        DataFrame        Shape: (522, 11)
batting_team           str              Sri Lanka
bowler                 str              AL Phehlukwayo
bowler_runs            int              1
bowler_wickets         int              0
bowling_data           DataFrame        Shape: (80736, 10)
bowling_match          DataFrame        Shape: (1819, 9)
bowling_matches        DataFrame        Shape: (78, 3)
bowling_records        list             n=80736
bowling_strength       DataFrame        Shape: (78, 11)
bowling_summary        DataFrame        Shape: (384, 9)
bowling_team           str              South Africa
candidate_2015         DataFrame        Shape: (19, 3)

In [115]:
team_venue_results = []

for _, row in master_data.iterrows():

    venue = row["standardized_venue"]
    team1 = row["team1"]
    team2 = row["team2"]
    winner = row["winner"]

    # Team 1
    team_venue_results.append({
        "standardized_venue": venue,
        "team": team1,
        "opponent": team2,
        "result": "Win" if winner == team1 else "Loss"
    })

    # Team 2
    team_venue_results.append({
        "standardized_venue": venue,
        "team": team2,
        "opponent": team1,
        "result": "Win" if winner == team2 else "Loss"
    })

team_venue_results = pd.DataFrame(team_venue_results)

team_venue_results.head(20)

,standardized_venue,team,opponent,result
0,Harare Sports Club,Zimbabwe,India,Loss
1,Harare Sports Club,India,Zimbabwe,Win
2,Harare Sports Club,Zimbabwe,India,Loss
3,Harare Sports Club,India,Zimbabwe,Win
4,Harare Sports Club,Zimbabwe,India,Loss
5,Harare Sports Club,India,Zimbabwe,Win
6,Harare Sports Club,Zimbabwe,Sri Lanka,Loss
7,Harare Sports Club,Sri Lanka,Zimbabwe,Win
8,Harare Sports Club,Sri Lanka,West Indies,Loss
9,Harare Sports Club,West Indies,Sri Lanka,Win


In [116]:
team_venue_results_summary = (
    team_venue_results
    .groupby(["standardized_venue", "team"], as_index=False)
    .agg(
        matches=("result", "count"),
        wins=("result", lambda x: (x == "Win").sum()),
        losses=("result", lambda x: (x == "Loss").sum())
    )
)

team_venue_results_summary["win_percentage"] = (
    team_venue_results_summary["wins"] /
    team_venue_results_summary["matches"]
) * 100

team_venue_results_summary.head(20)

,standardized_venue,team,matches,wins,losses,win_percentage
0,Boland Park,Australia,1,0,1,0.000000
1,Boland Park,Bangladesh,1,0,1,0.000000
2,Boland Park,India,3,1,2,33.333333
3,Boland Park,Pakistan,1,1,0,100.000000
4,Boland Park,South Africa,7,5,2,71.428571
5,Boland Park,Zimbabwe,1,0,1,0.000000
6,Buffalo Park,Bangladesh,1,0,1,0.000000
7,Buffalo Park,South Africa,3,2,1,66.666667
8,Buffalo Park,West Indies,2,1,1,50.000000
9,Harare Sports Club,Bangladesh,9,5,4,55.555556


In [117]:
print("Rows:", len(team_venue_results_summary))
print("Venues:", team_venue_results_summary["standardized_venue"].nunique())
print("Teams:", team_venue_results_summary["team"].nunique())

Rows: 78
Venues: 11
Teams: 20


In [120]:
historical_venues = set(
    team_venue_results_summary["standardized_venue"]
)

official_venue_names = set(
    official_venues["venue"]
)

print("Historical venues:")
print(sorted(historical_venues))

print("\nOfficial venues:")
print(sorted(official_venue_names))

print("\nOfficial venue without historical data:")
print(official_venue_names - historical_venues)

Historical venues:
['Boland Park', 'Buffalo Park', 'Harare Sports Club', 'Kingsmead', 'Mangaung Oval', 'Namibia Cricket Ground', 'Newlands', 'Queens Sports Club', "St George's Park", 'SuperSport Park', 'Wanderers Stadium']

Official venues:
['Boland Park', 'Buffalo Park', 'Harare Sports Club', 'Kingsmead', 'Mangaung Oval', 'Mosi-oa-Tunya International Cricket Stadium', 'Namibia Cricket Ground', 'Newlands', 'Queens Sports Club', "St George's Park", 'SuperSport Park', 'Wanderers Stadium']

Official venue without historical data:
{'Mosi-oa-Tunya International Cricket Stadium'}


In [119]:
print(official_venues.columns.tolist())
print(official_venues)

['venue', 'city', 'country']
                                          venue            city       country
0                             Wanderers Stadium    Johannesburg  South Africa
1                               SuperSport Park         Tshwane  South Africa
2                                      Newlands       Cape Town  South Africa
3                                     Kingsmead          Durban  South Africa
4                              St George's Park        Gqeberha  South Africa
5                                 Mangaung Oval    Bloemfontein  South Africa
6                                   Boland Park           Paarl  South Africa
7                                  Buffalo Park     East London  South Africa
8                            Harare Sports Club          Harare      Zimbabwe
9                            Queens Sports Club        Bulawayo      Zimbabwe
10  Mosi-oa-Tunya International Cricket Stadium  Victoria Falls      Zimbabwe
11                       Namibia Cr

In [121]:
print("BATTING:")
print(team_venue_batting.columns.tolist())

print("\nBOWLING:")
print(team_venue_bowling.columns.tolist())

print("\nRESULTS:")
print(team_venue_results_summary.columns.tolist())

BATTING:
['standardized_venue', 'batting_team', 'runs', 'balls_faced', 'fours', 'sixes', 'dismissals', 'batting_average', 'strike_rate', 'runs_per_match', 'innings', 'runs_per_innings']

BOWLING:
['standardized_venue', 'bowling_team', 'legal_deliveries', 'runs_conceded', 'wickets', 'economy', 'bowling_average', 'bowling_strike_rate']

RESULTS:
['standardized_venue', 'team', 'matches', 'wins', 'losses', 'win_percentage']


In [122]:
batting_master = team_venue_batting.rename(
    columns={"batting_team": "team"}
).copy()

bowling_master = team_venue_bowling.rename(
    columns={"bowling_team": "team"}
).copy()

results_master = team_venue_results_summary.copy()

In [123]:
batting_master = batting_master[
    [
        "standardized_venue",
        "team",
        "runs",
        "balls_faced",
        "fours",
        "sixes",
        "dismissals",
        "batting_average",
        "strike_rate",
        "runs_per_match",
        "innings",
        "runs_per_innings"
    ]
]

bowling_master = bowling_master[
    [
        "standardized_venue",
        "team",
        "legal_deliveries",
        "runs_conceded",
        "wickets",
        "economy",
        "bowling_average",
        "bowling_strike_rate"
    ]
]

In [124]:
team_venue_master = pd.merge(
    batting_master,
    bowling_master,
    on=["standardized_venue", "team"],
    how="outer"
)

In [125]:
team_venue_master = pd.merge(
    team_venue_master,
    results_master,
    on=["standardized_venue", "team"],
    how="outer"
)

In [126]:
print("Rows:", len(team_venue_master))
print("Venues:", team_venue_master["standardized_venue"].nunique())
print("Teams:", team_venue_master["team"].nunique())

team_venue_master.head(20)

Rows: 78
Venues: 11
Teams: 20


,standardized_venue,team,runs,balls_faced,fours,sixes,dismissals,batting_average,strike_rate,runs_per_match,...,legal_deliveries,runs_conceded,wickets,economy,bowling_average,bowling_strike_rate,matches,wins,losses,win_percentage
0,Boland Park,Australia,210.0,271.0,13.0,0.0,10.0,21.000000,77.490775,NaN,...,300,285,7,5.700000,40.714286,42.857143,1,0,1,0.000000
1,Boland Park,Bangladesh,236.0,287.0,21.0,3.0,10.0,23.600000,82.229965,NaN,...,300,351,6,7.020000,58.500000,50.000000,1,0,1,0.000000
2,Boland Park,India,793.0,903.0,66.0,14.0,22.0,36.045455,87.818383,NaN,...,864,790,17,5.486111,46.470588,50.823529,3,1,2,33.333333
3,Boland Park,Pakistan,229.0,297.0,17.0,5.0,7.0,32.714286,77.104377,NaN,...,300,235,9,4.700000,26.111111,33.333333,1,1,0,100.000000
4,Boland Park,South Africa,1838.0,2045.0,142.0,28.0,45.0,40.844444,89.877751,NaN,...,2052,1764,59,5.157895,29.898305,34.779661,7,5,2,71.428571
5,Boland Park,Zimbabwe,218.0,297.0,28.0,1.0,10.0,21.800000,73.400673,NaN,...,275,231,6,5.040000,38.500000,45.833333,1,0,1,0.000000
6,Buffalo Park,Bangladesh,161.0,245.0,21.0,0.0,10.0,16.100000,65.714286,NaN,...,300,366,6,7.320000,61.000000,50.000000,1,0,1,0.000000
7,Buffalo Park,South Africa,758.0,699.0,76.0,18.0,17.0,44.588235,108.440629,NaN,...,746,614,28,4.938338,21.928571,26.642857,3,2,1,66.666667
8,Buffalo Park,West Indies,427.0,504.0,36.0,15.0,18.0,23.722222,84.722222,NaN,...,398,405,11,6.105528,36.818182,36.181818,2,1,1,50.000000
9,Harare Sports Club,Bangladesh,2057.0,2493.0,190.0,30.0,60.0,34.283333,82.511031,NaN,...,2348,1957,74,5.000852,26.445946,31.729730,9,5,4,55.555556


In [127]:
team_venue_master["runs_per_match"] = (
    team_venue_master["runs"] /
    team_venue_master["matches"]
)

team_venue_master["runs_per_innings"] = (
    team_venue_master["runs"] /
    team_venue_master["innings"]
)

In [128]:
team_venue_master[
    [
        "standardized_venue",
        "team",
        "matches",
        "innings",
        "runs",
        "runs_per_match",
        "runs_per_innings",
        "wickets",
        "economy",
        "win_percentage"
    ]
].head(20)

,standardized_venue,team,matches,innings,runs,runs_per_match,runs_per_innings,wickets,economy,win_percentage
0,Boland Park,Australia,1,1.0,210.0,210.000000,210.000000,7,5.700000,0.000000
1,Boland Park,Bangladesh,1,1.0,236.0,236.000000,236.000000,6,7.020000,0.000000
2,Boland Park,India,3,3.0,793.0,264.333333,264.333333,17,5.486111,33.333333
3,Boland Park,Pakistan,1,1.0,229.0,229.000000,229.000000,9,4.700000,100.000000
4,Boland Park,South Africa,7,7.0,1838.0,262.571429,262.571429,59,5.157895,71.428571
5,Boland Park,Zimbabwe,1,1.0,218.0,218.000000,218.000000,6,5.040000,0.000000
6,Buffalo Park,Bangladesh,1,1.0,161.0,161.000000,161.000000,6,7.320000,0.000000
7,Buffalo Park,South Africa,3,3.0,758.0,252.666667,252.666667,28,4.938338,66.666667
8,Buffalo Park,West Indies,2,2.0,427.0,213.500000,213.500000,11,6.105528,50.000000
9,Harare Sports Club,Bangladesh,9,9.0,2057.0,228.555556,228.555556,74,5.000852,55.555556


In [129]:
team_venue_master.isna().sum()

standardized_venue     0
team                   0
runs                   1
balls_faced            1
fours                  1
sixes                  1
dismissals             1
batting_average        2
strike_rate            1
runs_per_match         1
innings                1
runs_per_innings       1
legal_deliveries       0
runs_conceded          0
wickets                0
economy                0
bowling_average        0
bowling_strike_rate    0
matches                0
wins                   0
losses                 0
win_percentage         0
dtype: int64

In [130]:
missing_batting = team_venue_master[
    team_venue_master["runs"].isna()
]

missing_batting[
    [
        "standardized_venue",
        "team",
        "runs",
        "batting_average",
        "strike_rate",
        "innings",
        "matches",
        "wickets",
        "economy",
        "win_percentage"
    ]
]

,standardized_venue,team,runs,batting_average,strike_rate,innings,matches,wickets,economy,win_percentage
25,Kingsmead,England,NaN,NaN,NaN,NaN,1,2,6.088235,0.0


In [131]:
team_venue_master[
    team_venue_master["batting_average"].isna()
][
    [
        "standardized_venue",
        "team",
        "runs",
        "dismissals",
        "batting_average",
        "strike_rate",
        "innings",
        "matches"
    ]
]

,standardized_venue,team,runs,dismissals,batting_average,strike_rate,innings,matches
25,Kingsmead,England,NaN,NaN,NaN,NaN,NaN,1
63,SuperSport Park,Netherlands,11.0,0.0,NaN,91.666667,1.0,1


In [132]:
scoring_data = team_venue_master.copy()

# Batting metrics
scoring_data["runs_per_innings"] = (
    scoring_data["runs"] /
    scoring_data["innings"]
)

# Bowling metric
scoring_data["wickets_per_match"] = (
    scoring_data["wickets"] /
    scoring_data["matches"]
)

# Check the scoring columns
scoring_data[
    [
        "standardized_venue",
        "team",
        "runs_per_innings",
        "strike_rate",
        "fours",
        "sixes",
        "wickets_per_match",
        "economy",
        "bowling_average",
        "bowling_strike_rate",
        "win_percentage"
    ]
].head(20)

,standardized_venue,team,runs_per_innings,strike_rate,fours,sixes,wickets_per_match,economy,bowling_average,bowling_strike_rate,win_percentage
0,Boland Park,Australia,210.000000,77.490775,13.0,0.0,7.000000,5.700000,40.714286,42.857143,0.000000
1,Boland Park,Bangladesh,236.000000,82.229965,21.0,3.0,6.000000,7.020000,58.500000,50.000000,0.000000
2,Boland Park,India,264.333333,87.818383,66.0,14.0,5.666667,5.486111,46.470588,50.823529,33.333333
3,Boland Park,Pakistan,229.000000,77.104377,17.0,5.0,9.000000,4.700000,26.111111,33.333333,100.000000
4,Boland Park,South Africa,262.571429,89.877751,142.0,28.0,8.428571,5.157895,29.898305,34.779661,71.428571
5,Boland Park,Zimbabwe,218.000000,73.400673,28.0,1.0,6.000000,5.040000,38.500000,45.833333,0.000000
6,Buffalo Park,Bangladesh,161.000000,65.714286,21.0,0.0,6.000000,7.320000,61.000000,50.000000,0.000000
7,Buffalo Park,South Africa,252.666667,108.440629,76.0,18.0,9.333333,4.938338,21.928571,26.642857,66.666667
8,Buffalo Park,West Indies,213.500000,84.722222,36.0,15.0,5.500000,6.105528,36.818182,36.181818,50.000000
9,Harare Sports Club,Bangladesh,228.555556,82.511031,190.0,30.0,8.222222,5.000852,26.445946,31.729730,55.555556


In [133]:
print(
    scoring_data[
        [
            "runs_per_innings",
            "strike_rate",
            "fours",
            "sixes",
            "wickets_per_match",
            "economy",
            "bowling_average",
            "bowling_strike_rate",
            "win_percentage"
        ]
    ].isna().sum()
)

runs_per_innings       1
strike_rate            1
fours                  1
sixes                  1
wickets_per_match      0
economy                0
bowling_average        0
bowling_strike_rate    0
win_percentage         0
dtype: int64


In [134]:
scoring_data = team_venue_master.copy()

# Positive metrics: higher = better
scoring_data["runs_per_innings"] = (
    scoring_data["runs"] / scoring_data["innings"]
)

scoring_data["wickets_per_match"] = (
    scoring_data["wickets"] / scoring_data["matches"]
)

In [135]:
positive_metrics = [
    "runs_per_innings",
    "strike_rate",
    "fours",
    "sixes",
    "wickets_per_match",
    "win_percentage"
]

for col in positive_metrics:
    min_val = scoring_data[col].min()
    max_val = scoring_data[col].max()

    scoring_data[col + "_score"] = (
        (scoring_data[col] - min_val) /
        (max_val - min_val)
    ) * 100

In [136]:
negative_metrics = [
    "economy",
    "bowling_average",
    "bowling_strike_rate"
]

for col in negative_metrics:
    min_val = scoring_data[col].min()
    max_val = scoring_data[col].max()

    scoring_data[col + "_score"] = (
        (max_val - scoring_data[col]) /
        (max_val - min_val)
    ) * 100

In [137]:
scoring_data[
    [
        "standardized_venue",
        "team",

        "runs_per_innings_score",
        "strike_rate_score",
        "fours_score",
        "sixes_score",

        "wickets_per_match_score",
        "economy_score",
        "bowling_average_score",
        "bowling_strike_rate_score",

        "win_percentage_score"
    ]
].head(20)

,standardized_venue,team,runs_per_innings_score,strike_rate_score,fours_score,sixes_score,wickets_per_match_score,economy_score,bowling_average_score,bowling_strike_rate_score,win_percentage_score
0,Boland Park,Australia,57.020057,39.272536,1.111111,0.000000,62.500000,59.469530,87.778408,83.185448,0.000000
1,Boland Park,Bangladesh,64.469914,46.042807,1.919192,1.796407,50.000000,33.302937,78.947368,77.639752,0.000000
2,Boland Park,India,72.588348,54.026262,6.464646,8.383234,45.833333,63.709487,84.920264,77.000365,33.333333
3,Boland Park,Pakistan,62.464183,38.720539,1.515152,2.994012,87.500000,79.292706,95.029240,90.579710,100.000000
4,Boland Park,South Africa,72.083504,56.968215,14.141414,16.766467,80.357143,70.215778,93.148806,89.456785,71.428571
5,Boland Park,Zimbabwe,59.312321,33.429533,2.626263,0.598802,50.000000,72.552826,88.877855,80.874741,0.000000
6,Buffalo Park,Bangladesh,42.979943,22.448980,1.919192,0.000000,50.000000,27.355984,77.706058,77.639752,0.000000
7,Buffalo Park,South Africa,69.245463,83.486614,7.474747,10.778443,91.666667,74.568094,97.105972,95.774179,66.666667
8,Buffalo Park,West Indies,58.022923,49.603175,3.434343,8.982036,43.750000,51.430684,89.712919,88.368154,50.000000
9,Harare Sports Club,Bangladesh,62.336835,46.444330,18.989899,17.964072,77.777778,73.328868,94.862986,91.824744,55.555556


In [138]:
scoring_data["batting_score"] = (
    scoring_data["runs_per_innings_score"] * 0.40 +
    scoring_data["strike_rate_score"] * 0.30 +
    scoring_data["fours_score"] * 0.15 +
    scoring_data["sixes_score"] * 0.15
)

scoring_data["bowling_score"] = (
    scoring_data["wickets_per_match_score"] * 0.40 +
    scoring_data["economy_score"] * 0.25 +
    scoring_data["bowling_average_score"] * 0.20 +
    scoring_data["bowling_strike_rate_score"] * 0.15
)

scoring_data["venue_score"] = (
    scoring_data["win_percentage_score"]
)

scoring_data["overall_venue_score"] = (
    scoring_data["batting_score"] * 0.40 +
    scoring_data["bowling_score"] * 0.40 +
    scoring_data["venue_score"] * 0.20
)

In [139]:
scoring_data[
    [
        "standardized_venue",
        "team",
        "batting_score",
        "bowling_score",
        "venue_score",
        "overall_venue_score"
    ]
].sort_values(
    "overall_venue_score",
    ascending=False
).head(20)

,standardized_venue,team,batting_score,bowling_score,venue_score,overall_venue_score
46,Queens Sports Club,Pakistan,56.021295,93.189302,100.000000,79.684239
60,SuperSport Park,Bangladesh,46.302468,92.974618,100.000000,75.710835
62,SuperSport Park,India,37.698607,97.661396,100.000000,74.144002
10,Harare Sports Club,India,40.285165,94.272874,100.000000,73.823216
75,Wanderers Stadium,South Africa,63.324883,79.135761,71.428571,71.269972
7,Buffalo Park,South Africa,55.482148,89.096012,66.666667,71.164597
26,Kingsmead,India,46.628038,78.149214,100.000000,69.910901
3,Boland Park,Pakistan,37.278209,87.415981,100.000000,69.877676
43,Newlands,South Africa,50.933936,83.448985,77.777778,69.308724
19,Harare Sports Club,Sri Lanka,42.365686,88.370501,83.333333,68.961142


In [140]:
wc_venue_scores = scoring_data[
    scoring_data["team"].isin(wc2027_teams)
].copy()

print("Rows:", len(wc_venue_scores))
print("Teams:", wc_venue_scores["team"].nunique())
print("Venues:", wc_venue_scores["standardized_venue"].nunique())

Rows: 67
Teams: 13
Venues: 11


In [141]:
wc_venue_scores[
    [
        "standardized_venue",
        "team",
        "matches",
        "overall_venue_score"
    ]
].sort_values(
    ["team", "standardized_venue"]
).head(30)

,standardized_venue,team,matches,overall_venue_score
0,Boland Park,Australia,1,41.862933
24,Kingsmead,Australia,1,49.840067
32,Mangaung Oval,Australia,3,66.754037
39,Newlands,Australia,1,49.649391
52,St George's Park,Australia,1,34.997296
59,SuperSport Park,Australia,2,37.170146
69,Wanderers Stadium,Australia,2,43.090249
1,Boland Park,Bangladesh,1,38.367727
6,Buffalo Park,Bangladesh,1,31.296288
9,Harare Sports Club,Bangladesh,9,61.751417


In [142]:
team_venue_final = (
    wc_venue_scores
    .dropna(subset=["overall_venue_score"])
    .groupby("team")
    .apply(
        lambda x: pd.Series({
            "historical_venues": x["standardized_venue"].nunique(),
            "historical_matches": x["matches"].sum(),
            "weighted_venue_score": (
                (x["overall_venue_score"] * x["matches"]).sum()
                / x["matches"].sum()
            )
        }),
        include_groups=False
    )
    .reset_index()
)

In [143]:
team_venue_final = (
    pd.DataFrame({"team": wc2027_teams})
    .merge(
        team_venue_final,
        on="team",
        how="left"
    )
)

In [144]:
team_venue_final = team_venue_final.sort_values(
    "weighted_venue_score",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

team_venue_final

,team,historical_venues,historical_matches,weighted_venue_score
0,South Africa,8.0,71.0,66.280248
1,India,7.0,21.0,66.144584
2,Pakistan,8.0,19.0,65.256178
3,Scotland,3.0,8.0,58.152271
4,Bangladesh,5.0,14.0,57.467882
5,Zimbabwe,4.0,65.0,57.167204
6,Ireland,1.0,16.0,57.056556
7,West Indies,7.0,18.0,55.060353
8,New Zealand,3.0,5.0,54.869958
9,England,5.0,9.0,51.217088


In [145]:
team_venue_final["venue_rank"] = (
    team_venue_final["weighted_venue_score"]
    .rank(
        ascending=False,
        method="min",
        na_option="bottom"
    )
)

team_venue_final

,team,historical_venues,historical_matches,weighted_venue_score,venue_rank
0,South Africa,8.0,71.0,66.280248,1.0
1,India,7.0,21.0,66.144584,2.0
2,Pakistan,8.0,19.0,65.256178,3.0
3,Scotland,3.0,8.0,58.152271,4.0
4,Bangladesh,5.0,14.0,57.467882,5.0
5,Zimbabwe,4.0,65.0,57.167204,6.0
6,Ireland,1.0,16.0,57.056556,7.0
7,West Indies,7.0,18.0,55.060353,8.0
8,New Zealand,3.0,5.0,54.869958,9.0
9,England,5.0,9.0,51.217088,10.0


In [148]:
# Create all official venue × 2027 team combinations

official_venue_names = official_venues["venue"].tolist()

venue_team_combinations = pd.MultiIndex.from_product(
    [
        official_venue_names,
        wc2027_teams
    ],
    names=["standardized_venue", "team"]
).to_frame(index=False)

print("Rows:", len(venue_team_combinations))
print("Venues:", venue_team_combinations["standardized_venue"].nunique())
print("Teams:", venue_team_combinations["team"].nunique())

venue_team_combinations.head(15)

Rows: 168
Venues: 12
Teams: 14


,standardized_venue,team
0,Wanderers Stadium,Australia
1,Wanderers Stadium,Bangladesh
2,Wanderers Stadium,England
3,Wanderers Stadium,India
4,Wanderers Stadium,Ireland
5,Wanderers Stadium,New Zealand
6,Wanderers Stadium,Pakistan
7,Wanderers Stadium,Scotland
8,Wanderers Stadium,South Africa
9,Wanderers Stadium,Sri Lanka


In [147]:
print(official_venues.columns.tolist())

['venue', 'city', 'country']


In [150]:
print("team_venue_stats columns:")
print(team_venue_stats.columns.tolist())

print("\nteam_venue_stats shape:")
print(team_venue_stats.shape)

print("\nFirst 5 rows:")
display(team_venue_stats.head())

team_venue_stats columns:
['team', 'venue', 'matches', 'wins', 'losses', 'win_percentage', 'overall_win_percentage', 'adjusted_venue_win_percentage']

team_venue_stats shape:
(78, 8)

First 5 rows:


,team,venue,matches,wins,losses,win_percentage,overall_win_percentage,adjusted_venue_win_percentage
0,Australia,Boland Park,1,0,1,0.000000,18.181818,15.151515
1,Australia,Kingsmead,1,0,1,0.000000,18.181818,15.151515
2,Australia,Mangaung Oval,3,2,1,66.666667,18.181818,36.363636
3,Australia,Newlands,1,0,1,0.000000,18.181818,15.151515
4,Australia,St George's Park,1,0,1,0.000000,18.181818,15.151515


In [151]:
print("team_venue_batting:", team_venue_batting.columns.tolist())
print("team_venue_bowling:", team_venue_bowling.columns.tolist())
print("team_venue_results:", team_venue_results.columns.tolist())

team_venue_batting: ['standardized_venue', 'batting_team', 'runs', 'balls_faced', 'fours', 'sixes', 'dismissals', 'batting_average', 'strike_rate', 'runs_per_match', 'innings', 'runs_per_innings']
team_venue_bowling: ['standardized_venue', 'bowling_team', 'legal_deliveries', 'runs_conceded', 'wickets', 'economy', 'bowling_average', 'bowling_strike_rate']
team_venue_results: ['standardized_venue', 'team', 'opponent', 'result']


In [153]:
venue_batting_score = team_venue_batting.copy()

venue_batting_score["runs_per_innings_score"] = (
    venue_batting_score["runs_per_innings"]
    .rank(pct=True) * 100
)

venue_batting_score["strike_rate_score"] = (
    venue_batting_score["strike_rate"]
    .rank(pct=True) * 100
)

venue_batting_score["fours_score"] = (
    venue_batting_score["fours"]
    .rank(pct=True) * 100
)

venue_batting_score["sixes_score"] = (
    venue_batting_score["sixes"]
    .rank(pct=True) * 100
)

venue_batting_score["batting_score"] = (
    venue_batting_score["runs_per_innings_score"] * 0.40
    + venue_batting_score["strike_rate_score"] * 0.30
    + venue_batting_score["fours_score"] * 0.15
    + venue_batting_score["sixes_score"] * 0.15
)

venue_batting_score = venue_batting_score[
    [
        "standardized_venue",
        "batting_team",
        "batting_score"
    ]
].rename(
    columns={"batting_team": "team"}
)

venue_batting_score.head()

,standardized_venue,team,batting_score
0,Boland Park,Australia,23.701299
1,Boland Park,Bangladesh,40.519481
2,Boland Park,India,72.240260
3,Boland Park,Pakistan,35.746753
4,Boland Park,South Africa,79.025974


In [154]:
venue_bowling_score = team_venue_bowling.copy()

venue_bowling_score["wickets_per_match"] = (
    venue_bowling_score["wickets"]
    / venue_bowling_score.groupby("standardized_venue")["wickets"]
    .transform("sum")
)

venue_bowling_score["wickets_per_match_score"] = (
    venue_bowling_score["wickets_per_match"]
    .rank(pct=True) * 100
)

# Lower economy is better
venue_bowling_score["economy_score"] = (
    1 - venue_bowling_score["economy"].rank(pct=True)
) * 100

# Lower bowling average is better
venue_bowling_score["bowling_average_score"] = (
    1 - venue_bowling_score["bowling_average"].rank(pct=True)
) * 100

# Lower strike rate is better
venue_bowling_score["bowling_strike_rate_score"] = (
    1 - venue_bowling_score["bowling_strike_rate"].rank(pct=True)
) * 100

venue_bowling_score["bowling_score"] = (
    venue_bowling_score["wickets_per_match_score"] * 0.40
    + venue_bowling_score["economy_score"] * 0.25
    + venue_bowling_score["bowling_average_score"] * 0.20
    + venue_bowling_score["bowling_strike_rate_score"] * 0.15
)

venue_bowling_score = venue_bowling_score[
    [
        "standardized_venue",
        "bowling_team",
        "bowling_score"
    ]
].rename(
    columns={"bowling_team": "team"}
)

venue_bowling_score.head()

,standardized_venue,team,bowling_score
0,Boland Park,Australia,37.980769
1,Boland Park,Bangladesh,23.301282
2,Boland Park,India,50.064103
3,Boland Park,Pakistan,70.480769
4,Boland Park,South Africa,77.564103


In [155]:
venue_score = pd.merge(
    venue_batting_score,
    venue_bowling_score,
    on=["standardized_venue", "team"],
    how="outer"
)

venue_score["overall_venue_score"] = (
    venue_score["batting_score"] * 0.50
    + venue_score["bowling_score"] * 0.50
)

venue_score.head()

,standardized_venue,team,batting_score,bowling_score,overall_venue_score
0,Boland Park,Australia,23.701299,37.980769,30.841034
1,Boland Park,Bangladesh,40.519481,23.301282,31.910381
2,Boland Park,India,72.240260,50.064103,61.152181
3,Boland Park,Pakistan,35.746753,70.480769,53.113761
4,Boland Park,South Africa,79.025974,77.564103,78.295038


In [156]:
print(venue_score.columns.tolist())
print(venue_score.shape)

['standardized_venue', 'team', 'batting_score', 'bowling_score', 'overall_venue_score']
(78, 5)


In [157]:
# Official venue names
official_venue_names = official_venues["venue"].unique().tolist()

# Create every venue × 2027 team combination
venue_team_combinations = pd.MultiIndex.from_product(
    [
        official_venue_names,
        wc2027_teams
    ],
    names=["standardized_venue", "team"]
).to_frame(index=False)

print("Rows:", len(venue_team_combinations))
print("Venues:", venue_team_combinations["standardized_venue"].nunique())
print("Teams:", venue_team_combinations["team"].nunique())

display(venue_team_combinations.head(20))

Rows: 168
Venues: 12
Teams: 14


,standardized_venue,team
0,Wanderers Stadium,Australia
1,Wanderers Stadium,Bangladesh
2,Wanderers Stadium,England
3,Wanderers Stadium,India
4,Wanderers Stadium,Ireland
5,Wanderers Stadium,New Zealand
6,Wanderers Stadium,Pakistan
7,Wanderers Stadium,Scotland
8,Wanderers Stadium,South Africa
9,Wanderers Stadium,Sri Lanka


In [158]:
print("Official venues:")
print(sorted(official_venue_names))

print("\nHistorical venues:")
print(sorted(venue_score["standardized_venue"].unique()))

Official venues:
['Boland Park', 'Buffalo Park', 'Harare Sports Club', 'Kingsmead', 'Mangaung Oval', 'Mosi-oa-Tunya International Cricket Stadium', 'Namibia Cricket Ground', 'Newlands', 'Queens Sports Club', "St George's Park", 'SuperSport Park', 'Wanderers Stadium']

Historical venues:
['Boland Park', 'Buffalo Park', 'Harare Sports Club', 'Kingsmead', 'Mangaung Oval', 'Namibia Cricket Ground', 'Newlands', 'Queens Sports Club', "St George's Park", 'SuperSport Park', 'Wanderers Stadium']


In [159]:
venue_team_scores = venue_team_combinations.merge(
    venue_score,
    on=["standardized_venue", "team"],
    how="left"
)

print(venue_team_scores.shape)
display(venue_team_scores.head(20))

(168, 5)


,standardized_venue,team,batting_score,bowling_score,overall_venue_score
0,Wanderers Stadium,Australia,47.532468,33.237179,40.384824
1,Wanderers Stadium,Bangladesh,11.785714,20.384615,16.085165
2,Wanderers Stadium,England,63.214286,57.500000,60.357143
3,Wanderers Stadium,India,44.610390,58.717949,51.664169
4,Wanderers Stadium,Ireland,NaN,NaN,NaN
5,Wanderers Stadium,New Zealand,NaN,NaN,NaN
6,Wanderers Stadium,Pakistan,76.038961,61.089744,68.564352
7,Wanderers Stadium,Scotland,NaN,NaN,NaN
8,Wanderers Stadium,South Africa,87.662338,75.064103,81.363220
9,Wanderers Stadium,Sri Lanka,22.564935,16.602564,19.583750


In [160]:
# Check how many combinations have historical data
print("Total combinations:", len(venue_team_scores))
print("With historical score:", venue_team_scores["overall_venue_score"].notna().sum())
print("Without historical score:", venue_team_scores["overall_venue_score"].isna().sum())

display(
    venue_team_scores[
        venue_team_scores["overall_venue_score"].isna()
    ]
)

Total combinations: 168
With historical score: 66
Without historical score: 102


,standardized_venue,team,batting_score,bowling_score,overall_venue_score
4,Wanderers Stadium,Ireland,NaN,NaN,NaN
5,Wanderers Stadium,New Zealand,NaN,NaN,NaN
7,Wanderers Stadium,Scotland,NaN,NaN,NaN
11,Wanderers Stadium,Zimbabwe,NaN,NaN,NaN
12,Wanderers Stadium,Namibia,NaN,NaN,NaN
...,...,...,...,...,...
162,Namibia Cricket Ground,South Africa,NaN,NaN,NaN
163,Namibia Cricket Ground,Sri Lanka,NaN,NaN,NaN
164,Namibia Cricket Ground,West Indies,NaN,NaN,NaN
165,Namibia Cricket Ground,Zimbabwe,NaN,NaN,NaN


In [162]:
# Calculate team-level historical venue ranking

team_venue_ranking = (
    venue_score
    .groupby("team")
    .agg(
        historical_venues=("standardized_venue", "nunique"),
        weighted_venue_score=("overall_venue_score", "mean")
    )
    .reset_index()
)

# Add historical match counts from team_venue_stats
team_match_counts = (
    team_venue_stats
    .groupby("team")
    .agg(
        historical_matches=("matches", "sum")
    )
    .reset_index()
)

team_venue_ranking = team_venue_ranking.merge(
    team_match_counts,
    on="team",
    how="left"
)

# Rank teams
team_venue_ranking["venue_rank"] = (
    team_venue_ranking["weighted_venue_score"]
    .rank(method="min", ascending=False)
)

# Add all 2027 teams, including Afghanistan
team_venue_ranking = (
    pd.DataFrame({"team": wc2027_teams})
    .merge(
        team_venue_ranking,
        on="team",
        how="left"
    )
)

team_venue_ranking = team_venue_ranking.sort_values(
    "venue_rank",
    na_position="last"
).reset_index(drop=True)

display(team_venue_ranking)

,team,historical_venues,weighted_venue_score,historical_matches,venue_rank
0,South Africa,8.0,76.891702,71.0,1.0
1,Ireland,1.0,62.119547,16.0,2.0
2,India,7.0,61.021062,21.0,3.0
3,Scotland,3.0,59.386308,8.0,4.0
4,Pakistan,8.0,56.977632,19.0,5.0
5,Namibia,1.0,53.576424,4.0,6.0
6,England,6.0,52.083874,10.0,7.0
7,West Indies,7.0,47.341052,18.0,8.0
8,Australia,7.0,47.243768,11.0,9.0
9,Zimbabwe,4.0,45.521041,65.0,10.0


In [163]:
print("Rows:", len(team_venue_ranking))
print("Teams:", team_venue_ranking["team"].nunique())

display(team_venue_ranking)

Rows: 14
Teams: 14


,team,historical_venues,weighted_venue_score,historical_matches,venue_rank
0,South Africa,8.0,76.891702,71.0,1.0
1,Ireland,1.0,62.119547,16.0,2.0
2,India,7.0,61.021062,21.0,3.0
3,Scotland,3.0,59.386308,8.0,4.0
4,Pakistan,8.0,56.977632,19.0,5.0
5,Namibia,1.0,53.576424,4.0,6.0
6,England,6.0,52.083874,10.0,7.0
7,West Indies,7.0,47.341052,18.0,8.0
8,Australia,7.0,47.243768,11.0,9.0
9,Zimbabwe,4.0,45.521041,65.0,10.0


In [164]:
team_fallback = team_venue_ranking[
    ["team", "weighted_venue_score"]
].copy()

team_fallback = team_fallback.rename(
    columns={
        "weighted_venue_score": "team_fallback_score"
    }
)

display(team_fallback)

,team,team_fallback_score
0,South Africa,76.891702
1,Ireland,62.119547
2,India,61.021062
3,Scotland,59.386308
4,Pakistan,56.977632
5,Namibia,53.576424
6,England,52.083874
7,West Indies,47.341052
8,Australia,47.243768
9,Zimbabwe,45.521041


In [165]:
venue_team_scores = venue_team_scores.merge(
    team_fallback,
    on="team",
    how="left"
)

print("Shape:", venue_team_scores.shape)

display(
    venue_team_scores.head(20)
)

Shape: (168, 6)


,standardized_venue,team,batting_score,bowling_score,overall_venue_score,team_fallback_score
0,Wanderers Stadium,Australia,47.532468,33.237179,40.384824,47.243768
1,Wanderers Stadium,Bangladesh,11.785714,20.384615,16.085165,41.275142
2,Wanderers Stadium,England,63.214286,57.500000,60.357143,52.083874
3,Wanderers Stadium,India,44.610390,58.717949,51.664169,61.021062
4,Wanderers Stadium,Ireland,NaN,NaN,NaN,62.119547
5,Wanderers Stadium,New Zealand,NaN,NaN,NaN,42.436522
6,Wanderers Stadium,Pakistan,76.038961,61.089744,68.564352,56.977632
7,Wanderers Stadium,Scotland,NaN,NaN,NaN,59.386308
8,Wanderers Stadium,South Africa,87.662338,75.064103,81.363220,76.891702
9,Wanderers Stadium,Sri Lanka,22.564935,16.602564,19.583750,39.750428


In [166]:
display(
    venue_team_scores[
        venue_team_scores["team"] == "Afghanistan"
    ]
)

,standardized_venue,team,batting_score,bowling_score,overall_venue_score,team_fallback_score
13,Wanderers Stadium,Afghanistan,NaN,NaN,NaN,NaN
27,SuperSport Park,Afghanistan,NaN,NaN,NaN,NaN
41,Newlands,Afghanistan,NaN,NaN,NaN,NaN
55,Kingsmead,Afghanistan,NaN,NaN,NaN,NaN
69,St George's Park,Afghanistan,NaN,NaN,NaN,NaN
83,Mangaung Oval,Afghanistan,NaN,NaN,NaN,NaN
97,Boland Park,Afghanistan,NaN,NaN,NaN,NaN
111,Buffalo Park,Afghanistan,NaN,NaN,NaN,NaN
125,Harare Sports Club,Afghanistan,NaN,NaN,NaN,NaN
139,Queens Sports Club,Afghanistan,NaN,NaN,NaN,NaN


In [167]:
display(
    venue_team_scores[
        venue_team_scores["standardized_venue"]
        == "Mosi-oa-Tunya International Cricket Stadium"
    ]
)

,standardized_venue,team,batting_score,bowling_score,overall_venue_score,team_fallback_score
140,Mosi-oa-Tunya International Cricket Stadium,Australia,NaN,NaN,NaN,47.243768
141,Mosi-oa-Tunya International Cricket Stadium,Bangladesh,NaN,NaN,NaN,41.275142
142,Mosi-oa-Tunya International Cricket Stadium,England,NaN,NaN,NaN,52.083874
143,Mosi-oa-Tunya International Cricket Stadium,India,NaN,NaN,NaN,61.021062
144,Mosi-oa-Tunya International Cricket Stadium,Ireland,NaN,NaN,NaN,62.119547
145,Mosi-oa-Tunya International Cricket Stadium,New Zealand,NaN,NaN,NaN,42.436522
146,Mosi-oa-Tunya International Cricket Stadium,Pakistan,NaN,NaN,NaN,56.977632
147,Mosi-oa-Tunya International Cricket Stadium,Scotland,NaN,NaN,NaN,59.386308
148,Mosi-oa-Tunya International Cricket Stadium,South Africa,NaN,NaN,NaN,76.891702
149,Mosi-oa-Tunya International Cricket Stadium,Sri Lanka,NaN,NaN,NaN,39.750428


In [168]:
neutral_baseline = venue_score["overall_venue_score"].mean()

print("Neutral baseline:", neutral_baseline)

Neutral baseline: 50.38128538128539


In [169]:
venue_team_scores["final_venue_score"] = (
    venue_team_scores["overall_venue_score"]
    .fillna(venue_team_scores["team_fallback_score"])
    .fillna(neutral_baseline)
)

In [170]:
print(
    "Missing final scores:",
    venue_team_scores["final_venue_score"].isna().sum()
)

Missing final scores: 0


In [171]:
display(
    venue_team_scores[
        venue_team_scores["team"] == "Afghanistan"
    ][
        [
            "standardized_venue",
            "team",
            "final_venue_score"
        ]
    ]
)

,standardized_venue,team,final_venue_score
13,Wanderers Stadium,Afghanistan,50.381285
27,SuperSport Park,Afghanistan,50.381285
41,Newlands,Afghanistan,50.381285
55,Kingsmead,Afghanistan,50.381285
69,St George's Park,Afghanistan,50.381285
83,Mangaung Oval,Afghanistan,50.381285
97,Boland Park,Afghanistan,50.381285
111,Buffalo Park,Afghanistan,50.381285
125,Harare Sports Club,Afghanistan,50.381285
139,Queens Sports Club,Afghanistan,50.381285


In [172]:
display(
    venue_team_scores[
        venue_team_scores["standardized_venue"]
        == "Mosi-oa-Tunya International Cricket Stadium"
    ][
        [
            "standardized_venue",
            "team",
            "team_fallback_score",
            "final_venue_score"
        ]
    ]
)

,standardized_venue,team,team_fallback_score,final_venue_score
140,Mosi-oa-Tunya International Cricket Stadium,Australia,47.243768,47.243768
141,Mosi-oa-Tunya International Cricket Stadium,Bangladesh,41.275142,41.275142
142,Mosi-oa-Tunya International Cricket Stadium,England,52.083874,52.083874
143,Mosi-oa-Tunya International Cricket Stadium,India,61.021062,61.021062
144,Mosi-oa-Tunya International Cricket Stadium,Ireland,62.119547,62.119547
145,Mosi-oa-Tunya International Cricket Stadium,New Zealand,42.436522,42.436522
146,Mosi-oa-Tunya International Cricket Stadium,Pakistan,56.977632,56.977632
147,Mosi-oa-Tunya International Cricket Stadium,Scotland,59.386308,59.386308
148,Mosi-oa-Tunya International Cricket Stadium,South Africa,76.891702,76.891702
149,Mosi-oa-Tunya International Cricket Stadium,Sri Lanka,39.750428,39.750428


In [173]:
final_venue_ranking = (
    venue_team_scores
    .sort_values(
        ["team", "final_venue_score"],
        ascending=[True, False]
    )
    .copy()
)

final_venue_ranking["venue_rank_for_team"] = (
    final_venue_ranking
    .groupby("team")["final_venue_score"]
    .rank(
        method="first",
        ascending=False
    )
)

final_venue_ranking = final_venue_ranking.sort_values(
    ["team", "venue_rank_for_team"]
)

print("Rows:", len(final_venue_ranking))
print("Teams:", final_venue_ranking["team"].nunique())
print("Venues:", final_venue_ranking["standardized_venue"].nunique())

display(
    final_venue_ranking[
        [
            "team",
            "standardized_venue",
            "final_venue_score",
            "venue_rank_for_team"
        ]
    ].head(30)
)

Rows: 168
Teams: 14
Venues: 12


,team,standardized_venue,final_venue_score,venue_rank_for_team
13,Afghanistan,Wanderers Stadium,50.381285,1.0
27,Afghanistan,SuperSport Park,50.381285,2.0
41,Afghanistan,Newlands,50.381285,3.0
55,Afghanistan,Kingsmead,50.381285,4.0
69,Afghanistan,St George's Park,50.381285,5.0
83,Afghanistan,Mangaung Oval,50.381285,6.0
97,Afghanistan,Boland Park,50.381285,7.0
111,Afghanistan,Buffalo Park,50.381285,8.0
125,Afghanistan,Harare Sports Club,50.381285,9.0
139,Afghanistan,Queens Sports Club,50.381285,10.0


In [174]:
top_3_venues = (
    final_venue_ranking[
        final_venue_ranking["venue_rank_for_team"] <= 3
    ]
    .copy()
)

display(
    top_3_venues[
        [
            "team",
            "standardized_venue",
            "final_venue_score",
            "venue_rank_for_team"
        ]
    ]
)

,team,standardized_venue,final_venue_score,venue_rank_for_team
13,Afghanistan,Wanderers Stadium,50.381285,1.0
27,Afghanistan,SuperSport Park,50.381285,2.0
41,Afghanistan,Newlands,50.381285,3.0
70,Australia,Mangaung Oval,77.196137,1.0
42,Australia,Kingsmead,53.772061,2.0
28,Australia,Newlands,52.304779,3.0
15,Bangladesh,SuperSport Park,70.542791,1.0
113,Bangladesh,Harare Sports Club,63.768315,2.0
29,Bangladesh,Newlands,41.275142,3.0
72,England,Mangaung Oval,68.777056,1.0


In [177]:
print("team_venue_ranking columns:")
print(team_venue_ranking.columns.tolist())

print("\nShape:")
print(team_venue_ranking.shape)

display(team_venue_ranking.head())

team_venue_ranking columns:
['team', 'historical_venues', 'weighted_venue_score', 'historical_matches', 'venue_rank']

Shape:
(14, 5)


,team,historical_venues,weighted_venue_score,historical_matches,venue_rank
0,South Africa,8.0,76.891702,71.0,1.0
1,Ireland,1.0,62.119547,16.0,2.0
2,India,7.0,61.021062,21.0,3.0
3,Scotland,3.0,59.386308,8.0,4.0
4,Pakistan,8.0,56.977632,19.0,5.0


In [178]:
print("venue_score columns:")
print(venue_score.columns.tolist() if "venue_score" in globals() else "venue_score does not exist")

venue_score columns:
['standardized_venue', 'team', 'batting_score', 'bowling_score', 'overall_venue_score']


In [179]:
print("Variables containing 'venue':")

for name in dir():
    if "venue" in name.lower():
        obj = globals()[name]
        if hasattr(obj, "columns"):
            print(name, "->", obj.columns.tolist(), "Shape:", obj.shape)

Variables containing 'venue':
final_venue_ranking -> ['standardized_venue', 'team', 'batting_score', 'bowling_score', 'overall_venue_score', 'team_fallback_score', 'final_venue_score', 'venue_rank_for_team'] Shape: (168, 8)
host_country_venues -> ['venue', 'city'] Shape: (35, 2)
official_venues -> ['venue', 'city', 'country'] Shape: (12, 3)
team_venue -> ['team', 'venue', 'matches', 'wins', 'losses', 'no_results', 'win_percentage'] Shape: (78, 7)
team_venue_batting -> ['standardized_venue', 'batting_team', 'runs', 'balls_faced', 'fours', 'sixes', 'dismissals', 'batting_average', 'strike_rate', 'runs_per_match', 'innings', 'runs_per_innings'] Shape: (77, 12)
team_venue_bowling -> ['standardized_venue', 'bowling_team', 'legal_deliveries', 'runs_conceded', 'wickets', 'economy', 'bowling_average', 'bowling_strike_rate'] Shape: (78, 8)
team_venue_final -> ['team', 'historical_venues', 'historical_matches', 'weighted_venue_score', 'venue_rank'] Shape: (14, 5)
team_venue_master -> ['standardi

In [180]:
display(
    final_venue_ranking[
        [
            "team",
            "standardized_venue",
            "final_venue_score",
            "venue_rank_for_team"
        ]
    ]
    .sort_values(["team", "venue_rank_for_team"])
)

,team,standardized_venue,final_venue_score,venue_rank_for_team
13,Afghanistan,Wanderers Stadium,50.381285,1.0
27,Afghanistan,SuperSport Park,50.381285,2.0
41,Afghanistan,Newlands,50.381285,3.0
55,Afghanistan,Kingsmead,50.381285,4.0
69,Afghanistan,St George's Park,50.381285,5.0
...,...,...,...,...
151,Zimbabwe,Mosi-oa-Tunya International Cricket Stadium,45.521041,8.0
165,Zimbabwe,Namibia Cricket Ground,45.521041,9.0
137,Zimbabwe,Queens Sports Club,44.332751,10.0
81,Zimbabwe,Mangaung Oval,41.139694,11.0


In [181]:
top_3_venues = (
    final_venue_ranking[
        final_venue_ranking["venue_rank_for_team"] <= 3
    ]
    .sort_values(["team", "venue_rank_for_team"])
)

print("Rows:", len(top_3_venues))

display(
    top_3_venues[
        [
            "team",
            "standardized_venue",
            "final_venue_score",
            "venue_rank_for_team"
        ]
    ]
)

Rows: 42


,team,standardized_venue,final_venue_score,venue_rank_for_team
13,Afghanistan,Wanderers Stadium,50.381285,1.0
27,Afghanistan,SuperSport Park,50.381285,2.0
41,Afghanistan,Newlands,50.381285,3.0
70,Australia,Mangaung Oval,77.196137,1.0
42,Australia,Kingsmead,53.772061,2.0
28,Australia,Newlands,52.304779,3.0
15,Bangladesh,SuperSport Park,70.542791,1.0
113,Bangladesh,Harare Sports Club,63.768315,2.0
29,Bangladesh,Newlands,41.275142,3.0
72,England,Mangaung Oval,68.777056,1.0


In [182]:
final_venue_ranking["data_source"] = np.where(
    final_venue_ranking["overall_venue_score"].notna(),
    "Historical Venue Data",
    "Team Fallback"
)

In [190]:
display(
    final_venue_ranking[
        final_venue_ranking["team"] == "Afghanistan"
    ][
        [
            "team",
            "standardized_venue",
            "overall_venue_score",
            "team_fallback_score",
            "final_venue_score",
            "data_source"
        ]
    ]
)

,team,standardized_venue,overall_venue_score,team_fallback_score,final_venue_score,data_source
13,Afghanistan,Wanderers Stadium,NaN,52.740346,52.740346,Team Fallback
27,Afghanistan,SuperSport Park,NaN,52.740346,52.740346,Team Fallback
41,Afghanistan,Newlands,NaN,52.740346,52.740346,Team Fallback
55,Afghanistan,Kingsmead,NaN,52.740346,52.740346,Team Fallback
69,Afghanistan,St George's Park,NaN,52.740346,52.740346,Team Fallback
83,Afghanistan,Mangaung Oval,NaN,52.740346,52.740346,Team Fallback
97,Afghanistan,Boland Park,NaN,52.740346,52.740346,Team Fallback
111,Afghanistan,Buffalo Park,NaN,52.740346,52.740346,Team Fallback
125,Afghanistan,Harare Sports Club,NaN,52.740346,52.740346,Team Fallback
139,Afghanistan,Queens Sports Club,NaN,52.740346,52.740346,Team Fallback


In [184]:
print(
    final_venue_ranking[
        final_venue_ranking["team"] == "Afghanistan"
    ][
        [
            "team",
            "standardized_venue",
            "overall_venue_score",
            "team_fallback_score",
            "final_venue_score"
        ]
    ]
)

            team                           standardized_venue  \
13   Afghanistan                            Wanderers Stadium   
27   Afghanistan                              SuperSport Park   
41   Afghanistan                                     Newlands   
55   Afghanistan                                    Kingsmead   
69   Afghanistan                             St George's Park   
83   Afghanistan                                Mangaung Oval   
97   Afghanistan                                  Boland Park   
111  Afghanistan                                 Buffalo Park   
125  Afghanistan                           Harare Sports Club   
139  Afghanistan                           Queens Sports Club   
153  Afghanistan  Mosi-oa-Tunya International Cricket Stadium   
167  Afghanistan                       Namibia Cricket Ground   

     overall_venue_score  team_fallback_score  final_venue_score  
13                   NaN                  NaN          50.381285  
27                  

In [185]:
print(
    team_venue_ranking[
        team_venue_ranking["team"] == "Afghanistan"
    ]
)

           team  historical_venues  weighted_venue_score  historical_matches  \
13  Afghanistan                NaN                   NaN                 NaN   

    venue_rank  
13         NaN  


In [186]:
fallback_score = team_venue_ranking[
    team_venue_ranking["weighted_venue_score"].notna()
]["weighted_venue_score"].mean()

print("Fallback score:", fallback_score)

Fallback score: 52.740346325682864


In [187]:
final_venue_ranking["team_fallback_score"] = (
    final_venue_ranking["team"]
    .map(
        team_venue_ranking.set_index("team")["weighted_venue_score"]
    )
)

final_venue_ranking["team_fallback_score"] = (
    final_venue_ranking["team_fallback_score"]
    .fillna(fallback_score)
)

In [188]:
final_venue_ranking["final_venue_score"] = (
    final_venue_ranking["overall_venue_score"]
    .fillna(final_venue_ranking["team_fallback_score"])
)

In [189]:
display(
    final_venue_ranking[
        final_venue_ranking["team"] == "Afghanistan"
    ][
        [
            "standardized_venue",
            "team",
            "overall_venue_score",
            "team_fallback_score",
            "final_venue_score"
        ]
    ]
)

,standardized_venue,team,overall_venue_score,team_fallback_score,final_venue_score
13,Wanderers Stadium,Afghanistan,NaN,52.740346,52.740346
27,SuperSport Park,Afghanistan,NaN,52.740346,52.740346
41,Newlands,Afghanistan,NaN,52.740346,52.740346
55,Kingsmead,Afghanistan,NaN,52.740346,52.740346
69,St George's Park,Afghanistan,NaN,52.740346,52.740346
83,Mangaung Oval,Afghanistan,NaN,52.740346,52.740346
97,Boland Park,Afghanistan,NaN,52.740346,52.740346
111,Buffalo Park,Afghanistan,NaN,52.740346,52.740346
125,Harare Sports Club,Afghanistan,NaN,52.740346,52.740346
139,Queens Sports Club,Afghanistan,NaN,52.740346,52.740346


In [191]:
final_venue_ranking["final_venue_score"] = (
    final_venue_ranking["overall_venue_score"]
    .fillna(final_venue_ranking["team_fallback_score"])
)

In [192]:
display(
    final_venue_ranking[
        final_venue_ranking["team"] == "Afghanistan"
    ][
        [
            "team",
            "standardized_venue",
            "overall_venue_score",
            "team_fallback_score",
            "final_venue_score"
        ]
    ]
)

,team,standardized_venue,overall_venue_score,team_fallback_score,final_venue_score
13,Afghanistan,Wanderers Stadium,NaN,52.740346,52.740346
27,Afghanistan,SuperSport Park,NaN,52.740346,52.740346
41,Afghanistan,Newlands,NaN,52.740346,52.740346
55,Afghanistan,Kingsmead,NaN,52.740346,52.740346
69,Afghanistan,St George's Park,NaN,52.740346,52.740346
83,Afghanistan,Mangaung Oval,NaN,52.740346,52.740346
97,Afghanistan,Boland Park,NaN,52.740346,52.740346
111,Afghanistan,Buffalo Park,NaN,52.740346,52.740346
125,Afghanistan,Harare Sports Club,NaN,52.740346,52.740346
139,Afghanistan,Queens Sports Club,NaN,52.740346,52.740346


In [193]:
display(
    final_venue_ranking[
        final_venue_ranking["team"] == "India"
    ][
        [
            "team",
            "standardized_venue",
            "overall_venue_score",
            "team_fallback_score",
            "final_venue_score"
        ]
    ]
    .sort_values("final_venue_score", ascending=False)
)

,team,standardized_venue,overall_venue_score,team_fallback_score,final_venue_score
31,India,Newlands,76.647935,61.021062,76.647935
115,India,Harare Sports Club,65.638112,61.021062,65.638112
17,India,SuperSport Park,62.064186,61.021062,62.064186
87,India,Boland Park,61.152181,61.021062,61.152181
73,India,Mangaung Oval,NaN,61.021062,61.021062
101,India,Buffalo Park,NaN,61.021062,61.021062
129,India,Queens Sports Club,NaN,61.021062,61.021062
143,India,Mosi-oa-Tunya International Cricket Stadium,NaN,61.021062,61.021062
157,India,Namibia Cricket Ground,NaN,61.021062,61.021062
45,India,Kingsmead,55.565684,61.021062,55.565684


In [194]:
venue_recommendations = final_venue_ranking.copy()

venue_recommendations["recommendation_type"] = np.where(
    venue_recommendations["overall_venue_score"].notna(),
    "Historical Venue Performance",
    "Fallback - No Historical Venue Data"
)

In [195]:
display(
    venue_recommendations[
        [
            "team",
            "standardized_venue",
            "final_venue_score",
            "recommendation_type"
        ]
    ]
)

,team,standardized_venue,final_venue_score,recommendation_type
13,Afghanistan,Wanderers Stadium,52.740346,Fallback - No Historical Venue Data
27,Afghanistan,SuperSport Park,52.740346,Fallback - No Historical Venue Data
41,Afghanistan,Newlands,52.740346,Fallback - No Historical Venue Data
55,Afghanistan,Kingsmead,52.740346,Fallback - No Historical Venue Data
69,Afghanistan,St George's Park,52.740346,Fallback - No Historical Venue Data
...,...,...,...,...
151,Zimbabwe,Mosi-oa-Tunya International Cricket Stadium,45.521041,Fallback - No Historical Venue Data
165,Zimbabwe,Namibia Cricket Ground,45.521041,Fallback - No Historical Venue Data
137,Zimbabwe,Queens Sports Club,44.332751,Historical Venue Performance
81,Zimbabwe,Mangaung Oval,41.139694,Historical Venue Performance


In [197]:
print(final_venue_ranking.columns.tolist())

['standardized_venue', 'team', 'batting_score', 'bowling_score', 'overall_venue_score', 'team_fallback_score', 'final_venue_score', 'venue_rank_for_team', 'data_source']


In [199]:
top_3_venues = (
    final_venue_ranking[
        final_venue_ranking["venue_rank_for_team"] <= 3
    ]
    .sort_values(
        ["team", "venue_rank_for_team"]
    )
)

print("Rows:", len(top_3_venues))
print("Teams:", top_3_venues["team"].nunique())

display(
    top_3_venues[
        [
            "team",
            "standardized_venue",
            "final_venue_score",
            "venue_rank_for_team",
            "data_source"
        ]
    ]
)

Rows: 42
Teams: 14


,team,standardized_venue,final_venue_score,venue_rank_for_team,data_source
13,Afghanistan,Wanderers Stadium,52.740346,1,Team Fallback
27,Afghanistan,SuperSport Park,52.740346,2,Team Fallback
41,Afghanistan,Newlands,52.740346,3,Team Fallback
70,Australia,Mangaung Oval,77.196137,1,Historical Venue Data
42,Australia,Kingsmead,53.772061,2,Historical Venue Data
28,Australia,Newlands,52.304779,3,Historical Venue Data
15,Bangladesh,SuperSport Park,70.542791,1,Historical Venue Data
113,Bangladesh,Harare Sports Club,63.768315,2,Historical Venue Data
29,Bangladesh,Newlands,41.275142,3,Team Fallback
72,England,Mangaung Oval,68.777056,1,Historical Venue Data


In [200]:
final_recommendations = (
    top_3_venues[
        top_3_venues["venue_rank_for_team"] == 1
    ]
    .sort_values("team")
    .reset_index(drop=True)
)

print("Rows:", len(final_recommendations))
print("Teams:", final_recommendations["team"].nunique())

display(
    final_recommendations[
        [
            "team",
            "standardized_venue",
            "final_venue_score",
            "data_source"
        ]
    ]
)

Rows: 14
Teams: 14


,team,standardized_venue,final_venue_score,data_source
0,Afghanistan,Wanderers Stadium,52.740346,Team Fallback
1,Australia,Mangaung Oval,77.196137,Historical Venue Data
2,Bangladesh,SuperSport Park,70.542791,Historical Venue Data
3,England,Mangaung Oval,68.777056,Historical Venue Data
4,India,Newlands,76.647935,Historical Venue Data
5,Ireland,Wanderers Stadium,62.119547,Team Fallback
6,Namibia,Wanderers Stadium,53.576424,Team Fallback
7,New Zealand,Harare Sports Club,54.812063,Historical Venue Data
8,Pakistan,Queens Sports Club,87.101648,Historical Venue Data
9,Scotland,Queens Sports Club,68.158508,Historical Venue Data


In [201]:
venue_recommendation_summary = (
    final_recommendations
    .groupby("standardized_venue")
    .agg(
        teams_recommended=("team", "count"),
        average_score=("final_venue_score", "mean")
    )
    .reset_index()
    .sort_values(
        ["teams_recommended", "average_score"],
        ascending=[False, False]
    )
)

display(venue_recommendation_summary)

,standardized_venue,teams_recommended,average_score
4,Queens Sports Club,3,73.509546
1,Harare Sports Club,3,59.213079
6,Wanderers Stadium,3,56.145439
2,Mangaung Oval,2,72.986597
0,Buffalo Park,1,85.632701
3,Newlands,1,76.647935
5,SuperSport Park,1,70.542791


In [202]:
recommendation_source_summary = (
    final_recommendations["data_source"]
    .value_counts()
    .reset_index()
)

recommendation_source_summary.columns = [
    "data_source",
    "team_count"
]

display(recommendation_source_summary)

,data_source,team_count
0,Historical Venue Data,11
1,Team Fallback,3


In [203]:
strongest_recommendation = (
    final_recommendations
    .sort_values("final_venue_score", ascending=False)
    .head(1)
)

display(
    strongest_recommendation[
        [
            "team",
            "standardized_venue",
            "final_venue_score",
            "data_source"
        ]
    ]
)

,team,standardized_venue,final_venue_score,data_source
8,Pakistan,Queens Sports Club,87.101648,Historical Venue Data


In [204]:
team_recommendation_ranking = (
    final_recommendations[
        [
            "team",
            "standardized_venue",
            "final_venue_score",
            "data_source"
        ]
    ]
    .sort_values("final_venue_score", ascending=False)
    .reset_index(drop=True)
)

team_recommendation_ranking["rank"] = (
    team_recommendation_ranking.index + 1
)

display(team_recommendation_ranking)

,team,standardized_venue,final_venue_score,data_source,rank
0,Pakistan,Queens Sports Club,87.101648,Historical Venue Data,1
1,South Africa,Buffalo Park,85.632701,Historical Venue Data,2
2,Australia,Mangaung Oval,77.196137,Historical Venue Data,3
3,India,Newlands,76.647935,Historical Venue Data,4
4,Bangladesh,SuperSport Park,70.542791,Historical Venue Data,5
5,England,Mangaung Oval,68.777056,Historical Venue Data,6
6,Scotland,Queens Sports Club,68.158508,Historical Venue Data,7
7,Sri Lanka,Queens Sports Club,65.268482,Historical Venue Data,8
8,West Indies,Harare Sports Club,63.232184,Historical Venue Data,9
9,Ireland,Wanderers Stadium,62.119547,Team Fallback,10


In [205]:
team_recommendation_ranking = (
    final_recommendations[
        [
            "team",
            "standardized_venue",
            "final_venue_score",
            "data_source"
        ]
    ]
    .sort_values("final_venue_score", ascending=False)
    .reset_index(drop=True)
)

team_recommendation_ranking["rank"] = (
    team_recommendation_ranking.index + 1
)

display(team_recommendation_ranking)

,team,standardized_venue,final_venue_score,data_source,rank
0,Pakistan,Queens Sports Club,87.101648,Historical Venue Data,1
1,South Africa,Buffalo Park,85.632701,Historical Venue Data,2
2,Australia,Mangaung Oval,77.196137,Historical Venue Data,3
3,India,Newlands,76.647935,Historical Venue Data,4
4,Bangladesh,SuperSport Park,70.542791,Historical Venue Data,5
5,England,Mangaung Oval,68.777056,Historical Venue Data,6
6,Scotland,Queens Sports Club,68.158508,Historical Venue Data,7
7,Sri Lanka,Queens Sports Club,65.268482,Historical Venue Data,8
8,West Indies,Harare Sports Club,63.232184,Historical Venue Data,9
9,Ireland,Wanderers Stadium,62.119547,Team Fallback,10


In [206]:
project_kpis = pd.DataFrame({
    "KPI": [
        "2027 Teams",
        "Official Venues",
        "Total Team-Venue Combinations",
        "Historical Data Teams",
        "Fallback Teams",
        "Historical Coverage %",
        "Fallback Coverage %",
        "Top Recommended Venue",
        "Highest Recommendation Score",
        "Highest Scoring Team"
    ],
    "Value": [
        final_recommendations["team"].nunique(),
        len(official_venues),
        len(venue_team_combinations),
        (final_recommendations["data_source"] == "Historical Venue Data").sum(),
        (final_recommendations["data_source"] == "Team Fallback").sum(),
        round(
            (final_recommendations["data_source"] == "Historical Venue Data").mean() * 100,
            2
        ),
        round(
            (final_recommendations["data_source"] == "Team Fallback").mean() * 100,
            2
        ),
        venue_recommendation_summary.iloc[0]["standardized_venue"],
        round(final_recommendations["final_venue_score"].max(), 2),
        final_recommendations.loc[
            final_recommendations["final_venue_score"].idxmax(),
            "team"
        ]
    ]
})

display(project_kpis)

,KPI,Value
0,2027 Teams,14
1,Official Venues,12
2,Total Team-Venue Combinations,168
3,Historical Data Teams,11
4,Fallback Teams,3
5,Historical Coverage %,78.57
6,Fallback Coverage %,21.43
7,Top Recommended Venue,Queens Sports Club
8,Highest Recommendation Score,87.1
9,Highest Scoring Team,Pakistan


In [207]:
final_recommendations.to_csv(
    "final_recommendations.csv",
    index=False
)

In [208]:
top_3_venues.to_csv(
    "top_3_venue_recommendations.csv",
    index=False
)

In [209]:
venue_recommendation_summary.to_csv(
    "venue_recommendation_summary.csv",
    index=False
)

In [210]:
final_venue_ranking.to_csv(
    "final_venue_ranking.csv",
    index=False
)

In [6]:
import os

project_path = r"D:\real project DA dataset\2027 wc"

print(os.listdir(project_path))

['2027wc.pbix', 'final_recommendations.csv', 'final_venue_ranking.csv', 'images.jpg', 'master_odi_2015_12venues.csv', 'odis_male_json', 'top_3_venue_recommendations.csv', 'venue_recommendation_summary.csv', 'wc.jpg']


In [7]:
csv_files = [
    f for f in os.listdir(project_path)
    if f.lower().endswith(".csv")
]

print("\n".join(csv_files))

final_recommendations.csv
final_venue_ranking.csv
master_odi_2015_12venues.csv
top_3_venue_recommendations.csv
venue_recommendation_summary.csv


In [8]:
import pandas as pd
import os

final_recommendations = pd.read_csv(
    os.path.join(project_path, "final_recommendations.csv")
)

final_venue_ranking = pd.read_csv(
    os.path.join(project_path, "final_venue_ranking.csv")
)

top_3_venue = pd.read_csv(
    os.path.join(project_path, "top_3_venue_recommendations.csv")
)

venue_recommendation_summary = pd.read_csv(
    os.path.join(project_path, "venue_recommendation_summary.csv")
)

In [9]:
print("final_recommendations:", final_recommendations.shape)
print("final_venue_ranking:", final_venue_ranking.shape)
print("top_3_venue:", top_3_venue.shape)
print("venue_recommendation_summary:", venue_recommendation_summary.shape)

final_recommendations: (14, 9)
final_venue_ranking: (168, 9)
top_3_venue: (42, 9)
venue_recommendation_summary: (7, 3)


In [10]:
print("\nTOP 3 VENUE COLUMNS:")
print(top_3_venue.columns.tolist())

print("\nFINAL VENUE RANKING COLUMNS:")
print(final_venue_ranking.columns.tolist())


TOP 3 VENUE COLUMNS:
['standardized_venue', 'team', 'batting_score', 'bowling_score', 'overall_venue_score', 'team_fallback_score', 'final_venue_score', 'venue_rank_for_team', 'data_source']

FINAL VENUE RANKING COLUMNS:
['standardized_venue', 'team', 'batting_score', 'bowling_score', 'overall_venue_score', 'team_fallback_score', 'final_venue_score', 'venue_rank_for_team', 'data_source']


In [11]:
display(
    final_venue_ranking[
        [
            "team",
            "standardized_venue",
            "batting_score",
            "bowling_score",
            "overall_venue_score",
            "team_fallback_score",
            "final_venue_score",
            "data_source"
        ]
    ].head(20)
)

,team,standardized_venue,batting_score,bowling_score,overall_venue_score,team_fallback_score,final_venue_score,data_source
0,Afghanistan,Wanderers Stadium,NaN,NaN,NaN,52.740346,52.740346,Team Fallback
1,Afghanistan,SuperSport Park,NaN,NaN,NaN,52.740346,52.740346,Team Fallback
2,Afghanistan,Newlands,NaN,NaN,NaN,52.740346,52.740346,Team Fallback
3,Afghanistan,Kingsmead,NaN,NaN,NaN,52.740346,52.740346,Team Fallback
4,Afghanistan,St George's Park,NaN,NaN,NaN,52.740346,52.740346,Team Fallback
5,Afghanistan,Mangaung Oval,NaN,NaN,NaN,52.740346,52.740346,Team Fallback
6,Afghanistan,Boland Park,NaN,NaN,NaN,52.740346,52.740346,Team Fallback
7,Afghanistan,Buffalo Park,NaN,NaN,NaN,52.740346,52.740346,Team Fallback
8,Afghanistan,Harare Sports Club,NaN,NaN,NaN,52.740346,52.740346,Team Fallback
9,Afghanistan,Queens Sports Club,NaN,NaN,NaN,52.740346,52.740346,Team Fallback


In [12]:
final_venue_ranking[
    [
        "batting_score",
        "bowling_score",
        "overall_venue_score",
        "team_fallback_score",
        "final_venue_score"
    ]
].describe()

,batting_score,bowling_score,overall_venue_score,team_fallback_score,final_venue_score
count,66.000000,67.000000,66.000000,168.000000,168.000000
mean,53.596025,52.401933,53.158978,52.740346,52.740346
std,22.925211,22.664471,17.367387,9.742512,12.756145
min,2.792208,1.121795,16.085165,39.750428,16.085165
25%,39.740260,37.211538,39.712059,45.521041,43.105020
50%,52.564935,55.256410,53.674242,52.412110,52.740346
75%,74.228896,68.926282,65.545704,59.386308,61.021062
max,94.220779,91.346154,87.101648,76.891702,87.101648


In [2]:
import os

project_path = r"D:\real project DA dataset\2027 wc"

csv_files = [
    f for f in os.listdir(project_path)
    if f.lower().endswith(".csv")
]

print("\n".join(csv_files))

final_recommendations.csv
final_venue_ranking.csv
master_odi_2015_12venues.csv
top_3_venue_recommendations.csv
venue_recommendation_summary.csv


In [3]:
files = os.listdir(project_path)

for f in files:
    if f.endswith((".ipynb", ".py")):
        print(f)

In [5]:
import os
import json
import pandas as pd

# ============================================================
# 1. PATHS
# ============================================================

json_folder = r"D:\real project DA dataset\2027 wc\odis_male_json"

# Your official 2027 World Cup venues
official_venues = [
    "Wanderers Stadium",
    "SuperSport Park",
    "Newlands",
    "Kingsmead",
    "St George's Park",
    "Mangaung Oval",
    "Boland Park",
    "Buffalo Park",
    "Harare Sports Club",
    "Queens Sports Club",
    "Mosi-oa-Tunya International Cricket Stadium",
    "Namibia Cricket Ground"
]

# ============================================================
# 2. VENUE STANDARDIZATION
# ============================================================

venue_mapping = {
    # Johannesburg
    "The Wanderers Stadium": "Wanderers Stadium",
    "Wanderers": "Wanderers Stadium",
    "Wanderers Stadium": "Wanderers Stadium",

    # Centurion
    "SuperSport Park": "SuperSport Park",

    # Cape Town
    "Newlands": "Newlands",
    "Newlands Cricket Ground": "Newlands",

    # Durban
    "Kingsmead": "Kingsmead",
    "Kingsmead Cricket Ground": "Kingsmead",

    # Gqeberha
    "St George's Park": "St George's Park",
    "St George's Park Cricket Ground": "St George's Park",

    # Bloemfontein
    "Mangaung Oval": "Mangaung Oval",
    "Mangaung Oval, Bloemfontein": "Mangaung Oval",

    # Paarl
    "Boland Park": "Boland Park",

    # East London
    "Buffalo Park": "Buffalo Park",

    # Harare
    "Harare Sports Club": "Harare Sports Club",

    # Bulawayo
    "Queens Sports Club": "Queens Sports Club",

    # Livingstone
    "Mosi-oa-Tunya International Cricket Stadium": 
        "Mosi-oa-Tunya International Cricket Stadium",

    # Namibia
    "Namibia Cricket Ground": "Namibia Cricket Ground"
}

# ============================================================
# 3. READ MATCH JSON FILES
# ============================================================

records = []

for filename in os.listdir(json_folder):

    if not filename.endswith(".json"):
        continue

    filepath = os.path.join(json_folder, filename)

    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        # ----------------------------------------------------
        # Extract info from Cricsheet-style JSON
        # ----------------------------------------------------

        info = data.get("info", {})

        gender = info.get("gender")
        match_type = info.get("match_type")

        # Only men's ODI
        if gender != "male":
            continue

        if match_type != "ODI":
            continue

        dates = info.get("dates", [])
        date = dates[0] if dates else None

        venue_info = info.get("venue")

        if not venue_info:
            continue

        standardized_venue = venue_mapping.get(
            venue_info,
            venue_info
        )

        # Only official 2027 venues
        if standardized_venue not in official_venues:
            continue

        teams = info.get("teams", [])

        if len(teams) != 2:
            continue

        records.append({
            "file": filename,
            "date": date,
            "venue": venue_info,
            "standardized_venue": standardized_venue,
            "team1": teams[0],
            "team2": teams[1]
        })

    except Exception as e:
        print(f"Error reading {filename}: {e}")

# ============================================================
# 4. CREATE MATCH DATAFRAME
# ============================================================

matches = pd.DataFrame(records)

print("Total historical matches at official venues:", len(matches))

print("\nColumns:")
print(matches.columns.tolist())

print("\nVenue counts:")
print(matches["standardized_venue"].value_counts())

Total historical matches at official venues: 270

Columns:
['file', 'date', 'venue', 'standardized_venue', 'team1', 'team2']

Venue counts:
standardized_venue
Harare Sports Club    113
Queens Sports Club     37
SuperSport Park        33
Kingsmead              23
St George's Park       20
Newlands               19
Mangaung Oval           8
Wanderers Stadium       7
Boland Park             5
Buffalo Park            5
Name: count, dtype: int64


In [8]:
# ============================================================
# 5. CREATE TEAM-VENUE MATCH RECORDS
# ============================================================

team_venue_records = []

for _, row in matches.iterrows():

    # Team 1
    team_venue_records.append({
        "team": row["team1"],
        "standardized_venue": row["standardized_venue"],
        "date": row["date"],
        "file": row["file"]
    })

    # Team 2
    team_venue_records.append({
        "team": row["team2"],
        "standardized_venue": row["standardized_venue"],
        "date": row["date"],
        "file": row["file"]
    })

team_venue_matches = pd.DataFrame(team_venue_records)

print(team_venue_matches.head())

print(
    "\nTeam-Venue rows:",
    len(team_venue_matches)
)

       team  standardized_venue        date          file
0  Zimbabwe  Harare Sports Club  2016-06-11  1007649.json
1     India  Harare Sports Club  2016-06-11  1007649.json
2  Zimbabwe  Harare Sports Club  2016-06-13  1007651.json
3     India  Harare Sports Club  2016-06-13  1007651.json
4  Zimbabwe  Harare Sports Club  2016-06-15  1007653.json

Team-Venue rows: 540


In [9]:
# ============================================================
# 6. HISTORICAL MATCH COUNT BY TEAM + VENUE
# ============================================================

historical_team_venue_summary = (
    team_venue_matches
    .groupby(
        ["team", "standardized_venue"],
        as_index=False
    )
    .agg(
        historical_matches=("file", "nunique")
    )
)

print(historical_team_venue_summary.head(20))

print(
    "\nTotal team-venue combinations:",
    len(historical_team_venue_summary)
)

          team  standardized_venue  historical_matches
0    Africa XI           Kingsmead                   1
1    Africa XI     SuperSport Park                   1
2      Asia XI           Kingsmead                   1
3      Asia XI     SuperSport Park                   1
4    Australia         Boland Park                   1
5    Australia  Harare Sports Club                   8
6    Australia           Kingsmead                   4
7    Australia       Mangaung Oval                   3
8    Australia            Newlands                   3
9    Australia    St George's Park                   4
10   Australia     SuperSport Park                   8
11  Bangladesh         Boland Park                   1
12  Bangladesh        Buffalo Park                   1
13  Bangladesh  Harare Sports Club                  14
14  Bangladesh  Queens Sports Club                  10
15     England        Buffalo Park                   1
16     England  Harare Sports Club                   2
17     Eng

In [10]:
# ============================================================
# 7. EXPORT
# ============================================================

output_path = r"D:\real project DA dataset\2027 wc\historical_team_venue_summary.csv"

historical_team_venue_summary.to_csv(
    output_path,
    index=False
)

print("Saved successfully:")
print(output_path)

Saved successfully:
D:\real project DA dataset\2027 wc\historical_team_venue_summary.csv


In [11]:
print(
    historical_team_venue_summary[
        historical_team_venue_summary["team"] == "Australia"
    ]
)

         team  standardized_venue  historical_matches
4   Australia         Boland Park                   1
5   Australia  Harare Sports Club                   8
6   Australia           Kingsmead                   4
7   Australia       Mangaung Oval                   3
8   Australia            Newlands                   3
9   Australia    St George's Park                   4
10  Australia     SuperSport Park                   8


In [1]:
# ============================================================
# 2027 WORLD CUP — FINAL 14 TEAMS
# ============================================================

world_cup_2027_teams = [
    'Australia',
    'Bangladesh',
    'England',
    'India',
    'Ireland',
    'New Zealand',
    'Pakistan',
    'Scotland',
    'South Africa',
    'Sri Lanka',
    'West Indies',
    'Zimbabwe',
    'Namibia',
    'Afghanistan'
]

print("2027 World Cup teams:", len(world_cup_2027_teams))
print(world_cup_2027_teams)

2027 World Cup teams: 14
['Australia', 'Bangladesh', 'England', 'India', 'Ireland', 'New Zealand', 'Pakistan', 'Scotland', 'South Africa', 'Sri Lanka', 'West Indies', 'Zimbabwe', 'Namibia', 'Afghanistan']


In [10]:
import os
import json
import pandas as pd

json_folder = r"D:\real project DA dataset\2027 wc\odis_male_json"

records = []

for filename in os.listdir(json_folder):

    if not filename.endswith(".json"):
        continue

    filepath = os.path.join(json_folder, filename)

    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        info = data.get("info", {})

        # Men's ODI only
        if info.get("gender") != "male":
            continue

        if info.get("match_type") != "ODI":
            continue

        teams = info.get("teams", [])

        if len(teams) != 2:
            continue

        dates = info.get("dates", [])
        date = dates[0] if dates else None

        venue = info.get("venue")

        if not venue:
            continue

        records.append({
            "file": filename,
            "date": date,
            "venue": venue,
            "team1": teams[0],
            "team2": teams[1]
        })

    except Exception as e:
        print(f"Error reading {filename}: {e}")

matches = pd.DataFrame(records)

print("Total men's ODI matches:", len(matches))
print("\nColumns:")
print(matches.columns.tolist())

print("\nFirst 5 rows:")
display(matches.head())

Total men's ODI matches: 2565

Columns:
['file', 'date', 'venue', 'team1', 'team2']

First 5 rows:


,file,date,venue,team1,team2
0,1000887.json,2017-01-13,"Brisbane Cricket Ground, Woolloongabba",Australia,Pakistan
1,1000889.json,2017-01-15,Melbourne Cricket Ground,Australia,Pakistan
2,1000891.json,2017-01-19,Western Australia Cricket Association Ground,Australia,Pakistan
3,1000893.json,2017-01-22,Sydney Cricket Ground,Australia,Pakistan
4,1000895.json,2017-01-26,Adelaide Oval,Australia,Pakistan


In [11]:
print("matches exists:", "matches" in globals())
print("Rows:", len(matches))
print("Date range:", matches["date"].min(), "to", matches["date"].max())

matches exists: True
Rows: 2565
Date range: 2002-06-27 to 2026-07-31


In [12]:
# ============================================================
# 2015 ONWARDS + 2027 WORLD CUP 14 TEAMS
# ============================================================

world_cup_2027_teams = [
    'Australia',
    'Bangladesh',
    'England',
    'India',
    'Ireland',
    'New Zealand',
    'Pakistan',
    'Scotland',
    'South Africa',
    'Sri Lanka',
    'West Indies',
    'Zimbabwe',
    'Namibia',
    'Afghanistan'
]

# Convert date
matches["date"] = pd.to_datetime(matches["date"])

# 2015 onwards
matches_2015 = matches[
    matches["date"].dt.year >= 2015
].copy()

# Only matches involving at least one of the 14 teams
matches_2015_wc = matches_2015[
    matches_2015["team1"].isin(world_cup_2027_teams) |
    matches_2015["team2"].isin(world_cup_2027_teams)
].copy()

print("Total men's ODI matches:", len(matches))

print("Matches from 2015 onwards:", len(matches_2015))

print("Matches involving 2027 teams:",
      len(matches_2015_wc))

Total men's ODI matches: 2565
Matches from 2015 onwards: 1258
Matches involving 2027 teams: 1099


In [13]:
# ============================================================
# TEAM + VENUE RECORDS
# ONLY THE 14 WORLD CUP TEAMS
# 2015 ONWARDS
# ============================================================

team_venue_records = []

for _, row in matches_2015_wc.iterrows():

    # Team 1
    if row["team1"] in world_cup_2027_teams:
        team_venue_records.append({
            "team": row["team1"],
            "standardized_venue": row["venue"],
            "date": row["date"],
            "file": row["file"]
        })

    # Team 2
    if row["team2"] in world_cup_2027_teams:
        team_venue_records.append({
            "team": row["team2"],
            "standardized_venue": row["venue"],
            "date": row["date"],
            "file": row["file"]
        })

team_venue_matches = pd.DataFrame(team_venue_records)

print("Team-venue records:", len(team_venue_matches))
print("Teams:", team_venue_matches["team"].nunique())

Team-venue records: 1996
Teams: 13


In [14]:
print(sorted(team_venue_matches["team"].unique()))

['Australia', 'Bangladesh', 'England', 'India', 'Ireland', 'Namibia', 'New Zealand', 'Pakistan', 'Scotland', 'South Africa', 'Sri Lanka', 'West Indies', 'Zimbabwe']


In [16]:
print(
    team_venue_matches[
        team_venue_matches["team"] == "Afghanistan"
    ]
)

Empty DataFrame
Columns: [team, standardized_venue, date, file]
Index: []


In [18]:
# ============================================================
# CREATE HISTORICAL TEAM-VENUE SUMMARY
# ============================================================

historical_team_venue_summary = (
    team_venue_matches
    .groupby(
        ["team", "standardized_venue"],
        as_index=False
    )
    .agg(
        historical_matches=("file", "nunique")
    )
)

print("Teams before Afghanistan:",
      historical_team_venue_summary["team"].nunique())

print("Rows before Afghanistan:",
      len(historical_team_venue_summary))

Teams before Afghanistan: 13
Rows before Afghanistan: 778


In [19]:
# ============================================================
# ADD AFGHANISTAN
# NO HISTORICAL VENUE DATA = 0 MATCHES
# ============================================================

official_venues = [
    "Wanderers Stadium",
    "SuperSport Park",
    "Newlands",
    "Kingsmead",
    "St George's Park",
    "Mangaung Oval",
    "Boland Park",
    "Buffalo Park",
    "Harare Sports Club",
    "Queens Sports Club",
    "Mosi-oa-Tunya International Cricket Stadium",
    "Namibia Cricket Ground"
]

afghanistan_rows = pd.DataFrame({
    "team": ["Afghanistan"] * len(official_venues),
    "standardized_venue": official_venues,
    "historical_matches": [0] * len(official_venues)
})

historical_team_venue_summary = pd.concat(
    [
        historical_team_venue_summary,
        afghanistan_rows
    ],
    ignore_index=True
)

In [20]:
print("Total teams:",
      historical_team_venue_summary["team"].nunique())

print("Total rows:",
      len(historical_team_venue_summary))

print("\nAfghanistan:")
display(
    historical_team_venue_summary[
        historical_team_venue_summary["team"] == "Afghanistan"
    ]
)

Total teams: 14
Total rows: 790

Afghanistan:


,team,standardized_venue,historical_matches
778,Afghanistan,Wanderers Stadium,0
779,Afghanistan,SuperSport Park,0
780,Afghanistan,Newlands,0
781,Afghanistan,Kingsmead,0
782,Afghanistan,St George's Park,0
783,Afghanistan,Mangaung Oval,0
784,Afghanistan,Boland Park,0
785,Afghanistan,Buffalo Park,0
786,Afghanistan,Harare Sports Club,0
787,Afghanistan,Queens Sports Club,0


In [21]:
print("Number of venues:",
      historical_team_venue_summary["standardized_venue"].nunique())

print("\nVenues:")
print(
    sorted(
        historical_team_venue_summary["standardized_venue"]
        .unique()
    )
)

Number of venues: 210

Venues:
['Adelaide Oval', 'Al Amerat Cricket Ground Oman Cricket (Ministry Turf 1)', 'Al Amerat Cricket Ground Oman Cricket (Ministry Turf 2)', 'Amini Park, Port Moresby', 'Arun Jaitley Stadium', 'Arun Jaitley Stadium, Delhi', 'Barabati Stadium', 'Barabati Stadium, Cuttack', 'Barsapara Cricket Stadium', 'Barsapara Cricket Stadium, Guwahati', 'Basin Reserve', 'Basin Reserve, Wellington', 'Bay Oval', 'Bay Oval, Mount Maunganui', 'Bellerive Oval', 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow', 'Bir Sreshtho Flight Lieutenant Matiur Rahman Stadium, Chattogram', 'Boland Park', 'Boland Park, Paarl', 'Brabourne Stadium', 'Bready Cricket Club, Magheramason', 'Brian Lara Stadium, Tarouba, Trinidad', 'Brisbane Cricket Ground, Woolloongabba', 'Buffalo Park', 'Buffalo Park, East London', 'Bulawayo Athletic Club', 'Cambusdoon New Ground, Ayr', 'Castle Avenue', 'Castle Avenue, Dublin', "Cazaly's Stadium, Cairns", 'Central Broward Regional Park Stadium

In [23]:
venues = sorted(
    historical_team_venue_summary["standardized_venue"].unique()
)

print("Total unique venues:", len(venues))

for v in venues:
    print(v)

Total unique venues: 210
Adelaide Oval
Al Amerat Cricket Ground Oman Cricket (Ministry Turf 1)
Al Amerat Cricket Ground Oman Cricket (Ministry Turf 2)
Amini Park, Port Moresby
Arun Jaitley Stadium
Arun Jaitley Stadium, Delhi
Barabati Stadium
Barabati Stadium, Cuttack
Barsapara Cricket Stadium
Barsapara Cricket Stadium, Guwahati
Basin Reserve
Basin Reserve, Wellington
Bay Oval
Bay Oval, Mount Maunganui
Bellerive Oval
Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow
Bir Sreshtho Flight Lieutenant Matiur Rahman Stadium, Chattogram
Boland Park
Boland Park, Paarl
Brabourne Stadium
Bready Cricket Club, Magheramason
Brian Lara Stadium, Tarouba, Trinidad
Brisbane Cricket Ground, Woolloongabba
Buffalo Park
Buffalo Park, East London
Bulawayo Athletic Club
Cambusdoon New Ground, Ayr
Castle Avenue
Castle Avenue, Dublin
Cazaly's Stadium, Cairns
Central Broward Regional Park Stadium Turf Ground
Choice Moosa Stadium, Pearland
Civil Service Cricket Club, Stormont
Civil Service Cri

In [22]:
official_venues = [
    "Wanderers Stadium",
    "SuperSport Park",
    "Newlands",
    "Kingsmead",
    "St George's Park",
    "Mangaung Oval",
    "Boland Park",
    "Buffalo Park",
    "Harare Sports Club",
    "Queens Sports Club",
    "Mosi-oa-Tunya International Cricket Stadium",
    "Namibia Cricket Ground"
]

In [24]:
# Show all raw venue names with their frequency

venue_counts = (
    matches_2015_wc["venue"]
    .value_counts()
    .reset_index()
)

venue_counts.columns = ["venue", "matches"]

display(venue_counts)

,venue,matches
0,Harare Sports Club,62
1,"Shere Bangla National Stadium, Mirpur",30
2,Dubai International Cricket Stadium,29
3,"R Premadasa Stadium, Colombo",28
4,Pallekele International Cricket Stadium,24
...,...,...
202,"Rajiv Gandhi International Stadium, Uppal",1
203,"Darren Sammy National Cricket Stadium, St Lucia",1
204,"St George's Park, Gqeberha",1
205,"Saxton Oval, Nelson",1


In [25]:
keywords = [
    "Wanderers",
    "SuperSport",
    "Newlands",
    "Kingsmead",
    "St George",
    "Mangaung",
    "Boland",
    "Buffalo",
    "Harare",
    "Queens",
    "Mosi",
    "Namibia"
]

for keyword in keywords:
    print("\n==============================")
    print(keyword)
    print("==============================")

    result = venue_counts[
        venue_counts["venue"].str.contains(
            keyword,
            case=False,
            na=False
        )
    ]

    display(result)


Wanderers


,venue,matches
13,"Wanderers Cricket Ground, Windhoek",13
62,"The Wanderers Stadium, Johannesburg",5
71,The Wanderers Stadium,5
94,New Wanderers Stadium,4
197,Wanderers Cricket Ground,1



SuperSport


,venue,matches
34,SuperSport Park,9
53,"SuperSport Park, Centurion",6



Newlands


...

,venue,matches
12,"Queens Sports Club, Bulawayo",13
14,Queens Sports Club,13



Mosi


,venue,matches



Namibia


,venue,matches
73,"Namibia Cricket Ground, Windhoek",5


In [26]:
venue_mapping = {

    # South Africa
    "The Wanderers Stadium, Johannesburg": "Wanderers Stadium",
    "The Wanderers Stadium": "Wanderers Stadium",
    "New Wanderers Stadium": "Wanderers Stadium",

    "SuperSport Park": "SuperSport Park",
    "SuperSport Park, Centurion": "SuperSport Park",

    "Newlands": "Newlands",
    "Newlands, Cape Town": "Newlands",

    "Kingsmead": "Kingsmead",

    "St George's Park": "St George's Park",
    "St George's Park, Gqeberha": "St George's Park",

    "Mangaung Oval": "Mangaung Oval",
    "Mangaung Oval, Bloemfontein": "Mangaung Oval",

    "Boland Park": "Boland Park",
    "Boland Park, Paarl": "Boland Park",

    "Buffalo Park": "Buffalo Park",
    "Buffalo Park, East London": "Buffalo Park",

    # Zimbabwe
    "Harare Sports Club": "Harare Sports Club",

    "Queens Sports Club": "Queens Sports Club",
    "Queens Sports Club, Bulawayo": "Queens Sports Club",

    # Namibia
    "Namibia Cricket Ground, Windhoek": "Namibia Cricket Ground"
}

In [27]:
team_venue_records = []

for _, row in matches_2015_wc.iterrows():

    standardized_venue = venue_mapping.get(row["venue"])

    # Keep only official venues
    if standardized_venue is None:
        continue

    # Team 1
    if row["team1"] in world_cup_2027_teams:
        team_venue_records.append({
            "team": row["team1"],
            "standardized_venue": standardized_venue,
            "date": row["date"],
            "file": row["file"]
        })

    # Team 2
    if row["team2"] in world_cup_2027_teams:
        team_venue_records.append({
            "team": row["team2"],
            "standardized_venue": standardized_venue,
            "date": row["date"],
            "file": row["file"]
        })

team_venue_matches = pd.DataFrame(team_venue_records)

print("Team-venue records:", len(team_venue_matches))
print("Teams:", team_venue_matches["team"].nunique())
print("Venues:", team_venue_matches["standardized_venue"].nunique())

Team-venue records: 302
Teams: 13
Venues: 11


In [28]:
historical_team_venue_summary = (
    team_venue_matches
    .groupby(
        ["team", "standardized_venue"],
        as_index=False
    )
    .agg(
        historical_matches=("file", "nunique")
    )
)

In [30]:
afghanistan_rows = pd.DataFrame({
    "team": ["Afghanistan"] * len(official_venues),
    "standardized_venue": official_venues,
    "historical_matches": [0] * len(official_venues)
})

historical_team_venue_summary = pd.concat(
    [
        historical_team_venue_summary,
        afghanistan_rows
    ],
    ignore_index=True
)

In [31]:
print("Teams:",
      historical_team_venue_summary["team"].nunique())

print("Venues:",
      historical_team_venue_summary["standardized_venue"].nunique())

print("Rows:",
      len(historical_team_venue_summary))

print("\nVenues:")
print(
    sorted(
        historical_team_venue_summary["standardized_venue"].unique()
    )
)

Teams: 14
Venues: 12
Rows: 92

Venues:
['Boland Park', 'Buffalo Park', 'Harare Sports Club', 'Kingsmead', 'Mangaung Oval', 'Mosi-oa-Tunya International Cricket Stadium', 'Namibia Cricket Ground', 'Newlands', 'Queens Sports Club', "St George's Park", 'SuperSport Park', 'Wanderers Stadium']


In [32]:
output_path = r"D:\real project DA dataset\2027 wc\historical_team_venue_summary.csv"

historical_team_venue_summary.to_csv(
    output_path,
    index=False
)

print("CSV created successfully:")
print(output_path)


CSV created successfully:
D:\real project DA dataset\2027 wc\historical_team_venue_summary.csv


In [34]:
import os

print(
    "File exists:",
    os.path.exists(output_path)
)

print(
    "Rows:",
    len(historical_team_venue_summary)
)

print(
    "Teams:",
    historical_team_venue_summary["team"].nunique()
)

print(
    "Venues:",
    historical_team_venue_summary["standardized_venue"].nunique()
)

File exists: True
Rows: 92
Teams: 14
Venues: 12
